# Case Study: Cyclistic Bike-Share

**How do annual members and casual riders use Cyclistic bikes differently?**<br>
*Google Data Analytics Professional Certificate – Course 8 Capstone*

**Shaine Meister**<br>
**March 28, 2026**

---


## Introduction

Cyclistic is a bike-share program in Chicago with more than 5,800 bicycles and 600 docking stations. The company offers traditional bikes as well as assistive options such as reclining bikes, hand tricycles, and cargo bikes. Until recently, Cyclistic’s marketing strategy focused on building general awareness with flexible pricing: single-ride passes, full-day passes, and annual memberships. Casual riders buy single-ride or full-day passes; annual members purchase yearly memberships.

Cyclistic’s finance team has shown that annual members are far more profitable than casual riders. The director of marketing, Lily Moreno, believes the company’s future success depends on converting casual riders into annual members. To design an effective marketing campaign, the team first needs clear answers to three guiding questions:

1. **How do annual members and casual riders use Cyclistic bikes differently?**  
2. **Why would casual riders buy Cyclistic annual memberships?**  
3. **How can Cyclistic use digital media to influence casual riders to become members?**

This notebook follows the official Google 6-step data analysis process (**Ask**, **Prepare**, **Process**, **Analyze**, **Share**, & **Act**) exactly. All analysis is performed in Python using only **pandas** and **numpy** modules. The data is the most recent 12 months of Cyclistic trip records (public Divvy dataset, March 2025 – February 2026). No personally identifiable information is present.

**Tools & Process**  
This notebook was developed in VS Code using pandas and numpy only. AI coding assistants (Grok + VS Code Copilot) were used as a research aid to generate and refine reusable helper functions and aggregation logic; as the data analyst, every cell was manually reviewed, tested for quality, and customized for the Cyclistic dataset.

**Personal Analytical Frameworks**  
Throughout the analysis I applied two frameworks I developed and use daily:  
- **[Technical Oscillation](https://x.com/shainemeister/status/2023320012149477721 "https://x.com/shainemeister/status/2023320012149477721")** – the iterative alternation between divergent (creative, generative) and convergent (logical, evaluative) thinking to balance imagination with feasibility.  
- **[Holographic Thinking](https://x.com/shainemeister/status/2034135203657179474 "https://x.com/shainemeister/status/2034135203657179474")** – the use of vivid, interactive mental simulations to visualize and test ideas as if they were tangible objects.  

These frameworks guided my mental process while reviewing data patterns, cleaning logic, and forming comparative insights in this project.

**Repository**  
The complete notebook, companion dataprocess files, and all supporting exports are available in the public GitHub repository: [case-study_bike-share-success](https://github.com/shainemeister/jupyter-notebooks/tree/main/case-study_bike-share-success "https://github.com/shainemeister/jupyter-notebooks/tree/main/case-study_bike-share-success")

## 1. Ask
**Business task**  
Design marketing strategies aimed at converting casual riders into annual members. The first step is to understand how annual members and casual riders use Cyclistic bikes differently so that targeted campaigns can be built.

**Key stakeholders**  
- Lily Moreno – Director of Marketing  
- Cyclistic marketing analytics team  
- Cyclistic executive team  

**Guiding questions (primary focus of this analysis)**  
- How do annual members versus casual riders differ in usage?  
- Why would casual riders buy a membership?  
- How can digital media convert casual riders into members?  

This analysis directly answers the first question and provides the foundation for answering the other two.

**Success criteria**  
Identify clear, actionable differences in ride length, day-of-week patterns, rideable type usage, and station preferences between the two rider groups.

## 2. Prepare
**Data location**  
Public Cyclistic (Divvy) trip data: [divvy data link](https://divvy-tripdata.s3.amazonaws.com/index.html "https://divvy-tripdata.s3.amazonaws.com/index.html")<br>
Weather Data: [Open-Meteo's](https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv "https://archive-api.open-meteo.com/v1/archive?latitude=41.65,42.10&longitude=-87.85,-87.40&start_date=2025-02-28&end_date=2026-02-28&hourly=temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,cloud_cover&timezone=America/Chicago&format=csv")<br>
Cleaned data source: [dataprocess](https://www.kaggle.com/code/shainemeister/case-study-1-gda-notebook-dataprocess "case-study-1-dga-notebook-dataprocess") - notebook processed from my [kaggle account](https://www.kaggle.com/code/shainemeister "https://www.kaggle.com/code/shainemeister").<br>

In this section, the notebook builds the raw trip dataset by validating the requested monthly Divvy files against the public S3 bucket, downloading ZIP archives when needed, extracting the monthly CSV files, and combining them into a single DataFrame.

**What this section does**
1. Validate the configured `start_yyyymm` to `end_yyyymm` month range against available Divvy S3 objects.
2. Download missing ZIP archives to a local cache and extract their CSV contents.
3. Concatenate all monthly CSV files into one consolidated dataset.
4. Standardize `started_at` as datetime and sort records chronologically.
5. Run a quick schema sanity check across extracted monthly files.

**How the data is organized** *(sample)* 
<div style="font-size: 10px;">

| `started_at` | `day_of_week` | `start_station_name` | `start_station_id` | `start-lat_vmap` | `start-lng_vmap` |
|---|---|---|---|---|---|
| Trip start timestamp | Num day of week (`0`=Mon, `6`=Sun) | Origin station name | Origin station identifier | Vector-mapped start latitude coordinate | Vector-mapped start longitude coordinate |

</div>

The combined trip dataset is expected to contain approximately 5-6 million rows, depending on the selected date range.

**ROCCC verification**  
- Reliable: Collected by Cyclistic's own system.  
- Original: First-party trip data.  
- Comprehensive: Covers every ride in the system.  
- Current: Uses the configured recent month range.  
- Cited: Licensed for analysis (Motivate International).  

**Licensing, privacy, and security**  
Data is public under the Divvy data license. No personally identifiable information is included, so privacy is preserved. Files are cached locally for reproducible reruns of the notebook.

**Prepare outputs produced here**  
- `df`: consolidated raw trip dataset with `started_at` parsed and sorted.  
- `schema_check`: quick cross-file schema QA summary.  

**Data integrity check**  
A quick preview of row structure and column consistency is displayed below before the notebook moves to the next section.

In [1]:
# ================================================================================
# Import required libraries
# ================================================================================
import zipfile
import numpy as np
import pandas as pd

In [2]:
# ================================================================================
# Load enriched trip data from ZIP archive
# Reads the first CSV found in the archive into df.
# ================================================================================
with zipfile.ZipFile('/mnt/RepoRetLabs/code/jupyter-notebooks/case-study_bike-share-success/202503-202602-divvy-tripdata-enriched.zip') as z:
    csv_files = [name for name in z.namelist() if name.lower().endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV file found inside the ZIP archive.")
    df = pd.read_csv(z.open(csv_files[0]))

# ================================================================================
# Quick shape and preview check
# ================================================================================
print(df.shape)
df.head()

(3480228, 30)


,started_at,day_of_week,start_station_name,start_station_id,start-lat_vmap,start-lng_vmap,end_station_name,end_station_id,end-lat_vmap,end-lng_vmap,...,weather_longitude,distance_miles,temperature_7ft_f,relative_humidity_7ft_pct,precipitation_in,rain_in,snowfall_in,wind_speed_33ft_mph,wind_direction_33ft_deg,cloud_cover_pct
0,2025-03-01 00:00:00,5,Lincoln Ave & Diversey Pkwy,TA1307000064,41.932225,-87.658625,Southport Ave & Waveland Ave,13235,41.948225,-87.664075,...,-87.37610,17.496295,32.72,69,0.0,0.0,0.0,27.029639,326,100
1,2025-03-01 00:00:00,5,Clark St & Berwyn Ave,KA1504000146,41.978025,-87.668575,Clark St & Schreiber Ave,KA1504000156,41.999225,-87.671225,...,-87.37610,16.405517,32.72,69,0.0,0.0,0.0,27.029639,326,100
2,2025-03-01 00:00:00,5,Blackstone Ave & 59th St,22004,41.787875,-87.590475,Public Rack - Woodlawn Ave & 63rd St N,951,41.780625,-87.596025,...,-87.78903,13.891121,35.78,63,0.0,0.0,0.0,10.501170,308,100
3,2025-03-01 00:00:00,5,Orleans St & Elm St,TA1306000006,41.902925,-87.637725,Loomis St & Jackson Blvd,13206,41.877925,-87.662025,...,-87.37610,17.886568,32.72,69,0.0,0.0,0.0,27.029639,326,100
4,2025-03-01 00:00:00,5,Orleans St & Elm St,TA1306000006,41.902925,-87.637725,Loomis St & Jackson Blvd,13206,41.877925,-87.662025,...,-87.37610,17.886568,32.72,69,0.0,0.0,0.0,27.029639,326,100


## 3. Process  
**Cleaning Summary** (already completed in companion notebook)  
- Removed negative ride lengths and extreme outliers (MAD z-score > 3.5 by rider type)  
- Created grid keys (`start-lat_vmap`) for stable geography  
- Merged hourly weather on grid + hour  
- Added simple flags: `is_weekend`, `rainy_day`, `temp_bin`  

**Weather metrics used in analysis**  
- `temperature_7ft_f` (temperature)  
- `wind_speed_33ft_mph` (wind speed)  
- `rain_in` (rain amount)  
- `cloud_cover_pct` (cloud cover)  

**Tools Used**  
- numpy  
- pandas (all steps)  
- zipfile (for direct import from zipped CSV)  

**Documentation**  
Full technical pipeline, validation counts, and outlier logs are in the companion notebook:  
`case-study-1-gda-notebook-dataprocess.ipynb` (linked in Appendix).

---

**What the code below does**  
The cell below carries the analysis directly from the cleaned dataset produced by Process.

1. **Select analysis dataframe** — Picks `df_filtered` (outlier-removed) if available, falls back to filtering `df` on the `ride_length_outlier` flag, or uses raw `df` as a last resort.  
2. **Validate required columns** — Checks that core fields and weather fields (`member_casual`, `ride_id`, `ride_length_seconds`, `temperature_7ft_f`, `wind_speed_33ft_mph`, `rain_in`, `cloud_cover_pct`) are present before running anything.  
3. **Build missing helper fields** — Recreates `is_weekend`, `rainy_day`, `temp_bin`, and `month` if needed.  
4. **Generate reusable summary tables and record lists for downstream analysis** — Produces structured outputs that are consumed in Analyze and Share without recomputing metrics in later cells.  
   - **Section 1: Usage differences** — Creates reusable table/list outputs grouped by rider type, temperature bin, and weekday/weekend with ride count and weather averages.  
   - **Section 2: Weather sensitivity** — Creates reusable table/list outputs grouped by rider type and rainy-day flag with ride count and weather averages.  
   - **Section 3: Seasonal patterns** — Creates reusable table/list outputs grouped by month and rider type with ride count and weather averages.  
   - **Section 4: Station summaries + top 5 per rider type** — Creates reusable station summary and top-5 outputs by rider group, with weather averages for each station.  
5. **Category reuse outputs** — Builds dynamic category tables and lists for all categorical-like columns so later cells can reuse category values quickly.

In [3]:
# ================================================================================
# Dataset selection
# Prefer the outlier-filtered dataset from Process.
# Falls back to filtering df on the outlier flag column,
# or raw df if neither is available.
# ================================================================================
if 'df_filtered' in globals():
    analysis_df = df_filtered.copy()
elif 'ride_length_outlier' in df.columns:
    analysis_df = df[~df['ride_length_outlier']].copy()
else:
    analysis_df = df.copy()

# ================================================================================
# Column validation
# Ensure required analysis + weather columns exist.
# ================================================================================
required_core = [
    'member_casual', 'ride_id', 'ride_length_seconds',
    'temperature_7ft_f', 'wind_speed_33ft_mph', 'rain_in', 'cloud_cover_pct'
]
missing_core = [c for c in required_core if c not in analysis_df.columns]
if missing_core:
    raise KeyError(f"Missing required columns in analysis_df: {missing_core}")

# ================================================================================
# Helper field fallbacks
# ================================================================================
if 'is_weekend' not in analysis_df.columns:
    if 'day_of_week' not in analysis_df.columns:
        raise KeyError("Missing 'is_weekend' and 'day_of_week'")
    analysis_df['is_weekend'] = analysis_df['day_of_week'].isin([5, 6])

if 'rainy_day' not in analysis_df.columns:
    analysis_df['rainy_day'] = analysis_df['rain_in'].fillna(0) > 0

if 'temp_bin' not in analysis_df.columns:
    analysis_df['temp_bin'] = pd.cut(
        analysis_df['temperature_7ft_f'],
        bins=[-np.inf, 45, 60, 75, np.inf],
        labels=['cold', 'cool', 'mild', 'warm']
    )

if 'month' not in analysis_df.columns:
    if 'started_at' not in analysis_df.columns:
        raise KeyError("Missing 'month' and 'started_at'; cannot build seasonal summary")
    analysis_df['started_at'] = pd.to_datetime(analysis_df['started_at'], errors='coerce')
    analysis_df['month'] = analysis_df['started_at'].dt.month

# ================================================================================
# Canonical time keys
# Derived once from started_at and attached to analysis_df.
# Used by reproducibility reports and complementary DataFrames.
# Columns: date, year, month_num, year_month, day_of_week_num,
#          day_of_week_name, hour_of_day.
# ================================================================================
if 'started_at' in analysis_df.columns:
    _started = pd.to_datetime(analysis_df['started_at'], errors='coerce')
    analysis_df['date']             = _started.dt.normalize()                   # midnight timestamp, date precision
    analysis_df['year']             = _started.dt.year                          # integer calendar year
    analysis_df['month_num']        = _started.dt.month                         # 1-12
    analysis_df['year_month']       = _started.dt.to_period('M').astype(str)    # e.g. '2025-03'
    _dow_src = (
        analysis_df['day_of_week']
        if 'day_of_week' in analysis_df.columns
        else _started.dt.dayofweek
    )
    analysis_df['day_of_week_num']  = _dow_src                                  # 0=Mon ... 6=Sun
    _dow_name_map_local = {
        0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
        4: 'Friday',  5: 'Saturday', 6: 'Sunday',
    }
    analysis_df['day_of_week_name'] = analysis_df['day_of_week_num'].map(_dow_name_map_local)
    analysis_df['hour_of_day']      = _started.dt.hour                          # 0-23

# ================================================================================
# Temporal integrity report
# Validates date coverage before any aggregation runs.
# Prints a WARNING for missing months but does not halt execution.
# ================================================================================
_missing_yms = []
if 'started_at' in analysis_df.columns:
    _ts           = pd.to_datetime(analysis_df['started_at'], errors='coerce')
    _date_min     = _ts.min()
    _date_max     = _ts.max()
    _null_ts      = int(_ts.isna().sum())
    _unique_dates = int(_ts.dt.date.nunique())
    _unique_years = int(_ts.dt.year.nunique())
    _unique_ym    = (
        int(analysis_df['year_month'].nunique())
        if 'year_month' in analysis_df.columns else 0
    )
    _expected_yms = sorted(
        str(p)
        for p in pd.period_range(
            pd.Period(_date_min, 'M'), pd.Period(_date_max, 'M'), freq='M'
        )
    )
    _actual_yms = (
        sorted(analysis_df['year_month'].dropna().unique().tolist())
        if 'year_month' in analysis_df.columns else []
    )
    _missing_yms = sorted(set(_expected_yms) - set(_actual_yms))
    temporal_integrity_report = pd.DataFrame([
        {'check': 'date_min',             'value': str(_date_min.date()),                                'status': 'OK'},
        {'check': 'date_max',             'value': str(_date_max.date()),                                'status': 'OK'},
        {'check': 'null_timestamps',      'value': str(_null_ts),                                        'status': 'OK' if _null_ts == 0 else 'WARN'},
        {'check': 'unique_dates',         'value': str(_unique_dates),                                   'status': 'OK'},
        {'check': 'unique_year_months',   'value': str(_unique_ym),                                      'status': 'OK' if _unique_ym >= 12 else 'WARN'},
        {'check': 'calendar_years_span',  'value': str(_unique_years),                                   'status': 'NOTE' if _unique_years > 1 else 'OK'},
        {'check': 'missing_year_months',  'value': ', '.join(_missing_yms) if _missing_yms else 'None', 'status': 'OK' if not _missing_yms else 'WARN'},
    ])

# ================================================================================
# Reusable category lists (key dimensions)
# ================================================================================
rider_type_list   = sorted(analysis_df['member_casual'].dropna().astype(str).unique().tolist())
temp_bin_list     = [str(x) for x in analysis_df['temp_bin'].dropna().unique().tolist()]
weekend_flag_list = sorted(analysis_df['is_weekend'].dropna().unique().tolist())
rainy_day_list    = sorted(analysis_df['rainy_day'].dropna().unique().tolist())
month_list        = sorted(analysis_df['month'].dropna().astype(int).unique().tolist())
year_month_list   = (
    sorted(analysis_df['year_month'].dropna().unique().tolist())
    if 'year_month' in analysis_df.columns else []
)

# ================================================================================
# Dynamic category outputs for all categorical-like columns
# ================================================================================
categorical_value_lists = {}
categorical_value_tables = {}
for col in analysis_df.columns:
    dtype = analysis_df[col].dtype
    if dtype == 'object' or dtype.name == 'category' or dtype == 'bool':
        values = sorted([v for v in analysis_df[col].dropna().unique().tolist()])
        categorical_value_lists[col] = values
        categorical_value_tables[col] = pd.DataFrame({
            'category_value': values
        })

# ================================================================================
# Shared aggregation specs (reused across all sections)
# weather_agg: avg of all 4 weather metrics
# base_agg: ride count + avg ride duration + weather_agg
# ================================================================================
weather_agg = {
    'avg_temperature_f':  ('temperature_7ft_f',   'mean'),
    'avg_wind_speed_mph': ('wind_speed_33ft_mph',  'mean'),
    'avg_rain_inches':    ('rain_in',              'mean'),
    'avg_cloud_cover_pct':('cloud_cover_pct',      'mean'),
}
base_agg = {
    'ride_count':       ('ride_id',              'count'),
    'avg_ride_seconds': ('ride_length_seconds',  'mean'),
    **weather_agg,
}

# ================================================================================
# SECTION 1: USAGE DIFFERENCES
# ================================================================================
usage_table = (
    analysis_df
    .groupby(['member_casual', 'temp_bin', 'is_weekend'], observed=False)
    .agg(**base_agg)
    .reset_index()
    .rename(columns={
        'member_casual': 'rider_type',
        'temp_bin': 'temperature_group',
        'is_weekend': 'weekend'
    })
    .round(2)
)
usage_list = usage_table.to_dict('records')

# ================================================================================
# SECTION 2: WEATHER SENSITIVITY
# ================================================================================
weather_impact_table = (
    analysis_df
    .groupby(['member_casual', 'rainy_day'])
    .agg(**base_agg)
    .reset_index()
    .rename(columns={'member_casual': 'rider_type'})
    .round(2)
)
weather_impact_list = weather_impact_table.to_dict('records')

# ================================================================================
# SECTION 3: SEASONAL PATTERNS
# ================================================================================
seasonal_table = (
    analysis_df
    .groupby(['month', 'member_casual'])
    .agg(**base_agg, rainy_day_rate=('rainy_day', 'mean'))
    .reset_index()
    .rename(columns={'member_casual': 'rider_type'})
    .round(2)
)
seasonal_list = seasonal_table.to_dict('records')

# ================================================================================
# SECTION 4: STATION SUMMARIES + TOP 5 PER RIDER TYPE
# ================================================================================
station_col = 'start_station_name' if 'start_station_name' in analysis_df.columns else 'start_station_id'

station_stats_table = (
    analysis_df
    .groupby([station_col, 'member_casual'])
    .agg(
        ride_count=('ride_id', 'count'),
        avg_month=('month', 'mean'),
        **weather_agg,
        rainy_day_rate=('rainy_day', 'mean')
    )
    .reset_index()
    .rename(columns={
        station_col: 'start_station',
        'member_casual': 'rider_type'
    })
    .round(2)
)
station_stats_list = station_stats_table.to_dict('records')

top5_stations_table = (
    station_stats_table
    .sort_values('ride_count', ascending=False)
    .groupby('rider_type')
    .head(5)
    .sort_values(['rider_type', 'ride_count'], ascending=[True, False])
    .reset_index(drop=True)
)
top5_stations_list = top5_stations_table.to_dict('records')

station_list = sorted(station_stats_table['start_station'].dropna().astype(str).unique().tolist())

# ================================================================================
# Complementary DataFrames  (research / reproducibility)
# Not required by core analysis outputs above.
# Documented here for audit traceability and reuse in Analyze.
# ================================================================================

# ================================================================================
# [R1] seasonal_full_table
# Year-aware seasonal summary grouped by year_month so months
# from different calendar years are never silently blended.
# ================================================================================
if 'year_month' in analysis_df.columns:
    seasonal_full_table = (
        analysis_df
        .groupby(['year_month', 'month_num', 'member_casual'])
        .agg(**base_agg, rainy_day_rate=('rainy_day', 'mean'))
        .reset_index()
        .rename(columns={'member_casual': 'rider_type'})
    )
    seasonal_full_table['month_name'] = (
        pd.PeriodIndex(seasonal_full_table['year_month'], freq='M').strftime('%B')
    )
    seasonal_full_table = (
        seasonal_full_table
        .sort_values(['year_month', 'rider_type'])
        .reset_index(drop=True)
    )
else:
    seasonal_full_table = pd.DataFrame()

# ================================================================================
# [R2] temperature_bin_balance
# Documents ride distribution across temperature bins.
# Flags bins with very sparse (<5%) or dominant (>60%) share.
# ================================================================================
_bin_counts = (
    analysis_df
    .groupby('temp_bin', observed=False)
    .agg(ride_count=('ride_id', 'count'))
    .reset_index()
)
_bin_total = _bin_counts['ride_count'].sum()
_bin_counts['pct_of_rides'] = (_bin_counts['ride_count'] / _bin_total * 100).round(2)
_bin_bounds = {'cold': '<=45 F', 'cool': '46-60 F', 'mild': '61-75 F', 'warm': '>75 F'}
_bin_counts['range_f']      = _bin_counts['temp_bin'].astype(str).map(_bin_bounds)
_bin_counts['balance_flag'] = _bin_counts['pct_of_rides'].apply(
    lambda p: 'SPARSE (<5%)' if p < 5 else ('DOMINANT (>60%)' if p > 60 else 'OK')
)
temperature_bin_balance = _bin_counts[
    ['temp_bin', 'range_f', 'ride_count', 'pct_of_rides', 'balance_flag']
].copy()

# ================================================================================
# Reusable variables (grouped)
# ================================================================================

# --- Primary analysis DataFrames ---
# analysis_df:              working analysis dataframe (cleaned/fallback)
# [1] usage_table:          usage summary by rider_type x temperature_group x weekend
# [2] weather_impact_table: weather sensitivity summary by rider_type x rainy_day
# [3] seasonal_table:       monthly seasonal summary by rider_type (numeric month key)
# [4] station_stats_table:  station-level summary by rider_type
# [4] top5_stations_table:  top 5 stations per rider_type by ride_count

# --- Complementary / reproducibility DataFrames ---
# temporal_integrity_report: temporal coverage checks (date range, null count, missing year-months)
# seasonal_full_table:        year-aware seasonal summary grouped by year_month (e.g. '2025-03')
#                             use instead of seasonal_table when year-accuracy is required
# temperature_bin_balance:    ride % per temp bin; flags SPARSE or DOMINANT bins

# --- Dynamic category helpers ---
# categorical_value_tables: dict of one-column DataFrames for each categorical-like column

# --- Primary lists ---
# [1] usage_list:          record-list version of usage_table
# [2] weather_impact_list: record-list version of weather_impact_table
# [3] seasonal_list:       record-list version of seasonal_table
# [4] station_stats_list:  record-list version of station_stats_table
# [4] top5_stations_list:  record-list version of top5_stations_table

# --- Category dimension lists ---
# rider_type_list:    unique rider groups
# temp_bin_list:      unique temperature groups
# weekend_flag_list:  unique weekend flags
# rainy_day_list:     unique rainy-day flags (from rain_in > 0)
# month_list:         unique months present in data (numeric, 1-12)
# year_month_list:    unique year_month period strings (e.g. ['2025-03', ..., '2026-02'])
# station_list:       unique stations used in station outputs

# --- Dynamic category lists ---
# categorical_value_lists: dict of unique values for each categorical-like column

# --- Canonical time columns added to analysis_df ---
# date:             midnight-normalised timestamp (date precision)
# year:             integer calendar year
# month_num:        integer month (1-12; mirrors existing 'month' column)
# year_month:       string period key (e.g. '2025-03')
# day_of_week_num:  integer 0=Mon ... 6=Sun
# day_of_week_name: full weekday name string
# hour_of_day:      integer 0-23

# --- Validation and config ---
# required_core: required analysis/weather columns
# missing_core:  missing columns list (empty when valid)
# station_col:   selected station source field (name first, ID fallback)
# weather_agg:   shared dict of 4 weather metric aggregations (reused in all sections)
# base_agg:      shared dict of ride_count + avg_ride_seconds + weather_agg

In [4]:
if 'temporal_integrity_report' in globals():
    print("\n[Integrity] Temporal coverage:")
    print(temporal_integrity_report.to_string(index=False))
    if _missing_yms:
        print(f"  WARNING: missing year-months detected: {_missing_yms}")

print(f"Analyzing rows: {len(analysis_df):,}")

print("\n[1] Usage differences")
print(usage_table.head(12).to_string(index=False))

print("\n[2] Weather sensitivity")
print(weather_impact_table.to_string(index=False))

print("\n[3] Seasonal patterns")
print(seasonal_table.head(12).to_string(index=False))

print("\n[4] Top 5 start stations by rider type")
for rider_type, rider_frame in top5_stations_table.groupby('rider_type'):
    print(f"\n  {rider_type.upper()}")
    print(rider_frame.to_string(index=False))

if not seasonal_full_table.empty:
    print("\n[R1] Year-aware seasonal summary (seasonal_full_table):")
    print(
        seasonal_full_table[[
            'year_month', 'month_name', 'rider_type',
            'ride_count', 'avg_ride_seconds', 'rainy_day_rate'
        ]].to_string(index=False)
    )

print("\n[R2] Temperature bin balance (temperature_bin_balance):")
print(temperature_bin_balance.to_string(index=False))


[Integrity] Temporal coverage:
              check      value status
           date_min 2025-03-01     OK
           date_max 2026-02-28     OK
    null_timestamps          0     OK
       unique_dates        365     OK
 unique_year_months         12     OK
calendar_years_span          2   NOTE
missing_year_months       None     OK
Analyzing rows: 3,340,737

[1] Usage differences
rider_type temperature_group  weekend  ride_count  avg_ride_seconds  avg_temperature_f  avg_wind_speed_mph  avg_rain_inches  avg_cloud_cover_pct
    casual              cold    False      107605            783.79              36.94               11.64              0.0                63.72
    casual              cold     True       64013           1009.52              37.75               11.35              0.0                48.66
    casual              cool    False      161276            921.12              52.82               11.56              0.0                56.08
    casual              cool     Tr

## 4. Analyze

This section converts the processed data into clear, comparative rider-behavior findings using simple pandas group-by and aggregation, then renders both technical outputs and a stakeholder-ready dynamic summary.

**Step-by-step process**

1. **Prepare and validate analysis inputs**  
   The script verifies required summary tables, makes local working copies, and standardizes rider labels (`casual`, `member`) for consistent comparisons.

2. **[1] Compare core usage behavior**  
   Calculates rider-level totals and weighted averages (ride duration, temperature, cloud cover).  
   Output focus: ride-length lift and baseline weather-condition differences between rider types.

3. **[2] Measure weather sensitivity**  
   Splits rides into rainy vs non-rainy conditions and measures changes in ride volume and ride duration.  
   Output focus: how strongly each rider type reacts to rain and cloudier conditions.

4. **[3] Identify seasonal timing windows**  
   Finds each rider group’s peak month, summarizes the May-September window, and checks rainy-day-rate variation.  
   Output focus: seasonal volume and ride-length patterns by rider type.

5. **[4] Locate priority start stations**  
   Ranks top stations by rider type and calculates top-5 concentration share.  
   Output focus: geographic concentration by rider type.

6. **Package reusable outputs for Share/Act**  
   Stores results in reusable structures:  
   - `analyze_key_metrics_table`  
   - `analyze_section_summaries`  
   - `analyze_key_metrics_list`

7. **Render final analysis outputs**  
   Prints standardized section summaries, audit/reproducibility tables, and complementary diagnostics for transparent validation.

8. **Generate dynamic markdown summary**  
   Builds and displays a metric-driven Analysis Summary block with sign-aware wording (for example warmer/cooler, longer/shorter, lower/higher cloud cover) so narrative text updates automatically when data changes.

**Why this matters**  
This workflow turns raw ride records into decision-ready comparative insights by clarifying:  
- **who** behaves differently (`casual` vs `member`),  
- **when** volume and ride length peak, and  
- **where** casual rides concentrate.

It also improves reproducibility by keeping technical outputs and stakeholder-facing narrative synchronized from the same computed metrics.

In [5]:
# ================================================================================
# SECTION 0: ANALYZE SCRIPT (SECTIONS 1-4)
# ================================================================================

# Ensure upstream summary tables exist before analysis starts.
required_tables = [
    'usage_table', 'weather_impact_table', 'seasonal_table',
    'station_stats_table', 'top5_stations_table'
]
missing_tables = [t for t in required_tables if t not in globals()]
if missing_tables:
    raise KeyError(f"Missing required table(s): {missing_tables}")

# Use local copies to avoid mutating shared upstream tables.
usage_an, weather_an, seasonal_an, station_stats_an, top5_an = (
    usage_table.copy(), weather_impact_table.copy(), seasonal_table.copy(),
    station_stats_table.copy(), top5_stations_table.copy()
)


# Normalize rider labels so all downstream filters are consistent.
def normalize_rider(series):
    """Normalize rider labels for stable comparisons (member/casual)."""
    return series.astype(str).str.strip().str.lower().str.replace(' ', '_', regex=False)


# Return percentages safely when denominator values can be zero.
def safe_pct(numerator, denominator):
    """Return percentage while guarding against divide-by-zero."""
    return (numerator / denominator * 100.0) if denominator else 0.0


# Compute weighted averages using ride_count as the weight column.
def weighted_mean(frame, value_col, weight_col='ride_count'):
    """Weighted mean helper used for rider-level rollups."""
    if frame.empty:
        return 0.0
    weights = frame[weight_col].astype(float)
    values  = frame[value_col].astype(float)
    weight_sum = weights.sum()
    return float((values * weights).sum() / weight_sum) if weight_sum else 0.0


# Shared output helpers keep all text blocks visually consistent.
OUTPUT_WIDTH = 72

def print_block_header(title, rule_char='='):
    """Print a standardized title block for console outputs."""
    rule = rule_char * OUTPUT_WIDTH
    print(f"\n{rule}")
    print(f"  {title}")
    print(rule)


# Render section headers and consistently formatted text tables.
def show_section(title, df, fmt=None):
    """Print a standardized section title and formatted DataFrame."""
    print_block_header(title, rule_char='-')
    display_df = df.copy()
    if fmt:
        for col, f in fmt.items():
            if col in display_df.columns:
                display_df[col] = display_df[col].apply(lambda v: f.format(v))
    print(display_df.to_string(index=False))


# ================================================================================
# SECTION 0a: SHARED CONFIG
# ================================================================================

# month_name_map maps month numbers to full month names.
# Static mapping keeps outputs deterministic across runs.
month_name_map = {i: pd.Timestamp(2000, i, 1).strftime('%B') for i in range(1, 13)}

# campaign_months_fixed is the fixed May-Sep campaign window.
# Keep this list stable for core seasonal comparisons.
# campaign_months remains an alias for downstream Share compatibility.
campaign_months_fixed = [5, 6, 7, 8, 9]
campaign_months       = campaign_months_fixed

# CAMPAIGN_TARGET_PCT sets the target casual volume share (%) for
# the data-derived campaign window in complementary analysis.
# Increase to broaden coverage, decrease to tighten focus.
CAMPAIGN_TARGET_PCT = 70.0

# Warn when fixed campaign months are absent from observed data.
_seasonal_an_months = set(seasonal_table['month'].dropna().astype(int).tolist())
_missing_campaign_months = [m for m in campaign_months_fixed if m not in _seasonal_an_months]

# Apply rider label normalization once across all local tables.
for frame in [usage_an, weather_an, seasonal_an, station_stats_an, top5_an]:
    frame['rider_type'] = normalize_rider(frame['rider_type'])

# Build rider_types once and reuse across sections.
rider_types = sorted(usage_an['rider_type'].dropna().unique().tolist())


# ================================================================================
# SECTION 1: USAGE DIFFERENCES
# ================================================================================

usage_rider_summary_rows = []
for rider in rider_types:
    rider_frame = usage_an[usage_an['rider_type'] == rider]
    usage_rider_summary_rows.append({
        'rider_type':          rider,
        'total_rides':         int(rider_frame['ride_count'].sum()),
        'avg_ride_seconds':    round(weighted_mean(rider_frame, 'avg_ride_seconds'), 2),
        'avg_temperature_f':   round(weighted_mean(rider_frame, 'avg_temperature_f'), 2),
        'avg_cloud_cover_pct': round(weighted_mean(rider_frame, 'avg_cloud_cover_pct'), 2),
    })

usage_rider_summary = pd.DataFrame(usage_rider_summary_rows)

# Retain explicit rider slices for downstream compatibility.
casual_usage = usage_rider_summary[usage_rider_summary['rider_type'] == 'casual']
member_usage = usage_rider_summary[usage_rider_summary['rider_type'] == 'member']

# Use an index lookup for cleaner scalar extraction by rider type.
_usage_idx = usage_rider_summary.set_index('rider_type')
casual_avg_ride  = float(_usage_idx.loc['casual', 'avg_ride_seconds']) if 'casual' in _usage_idx.index else 0.0
member_avg_ride  = float(_usage_idx.loc['member', 'avg_ride_seconds']) if 'member' in _usage_idx.index else 0.0
duration_lift_pct = round(safe_pct(casual_avg_ride - member_avg_ride, member_avg_ride), 2)
if {'casual', 'member'}.issubset(_usage_idx.index):
    temp_delta_f   = round(
        float(_usage_idx.loc['casual', 'avg_temperature_f']) - float(_usage_idx.loc['member', 'avg_temperature_f']), 2
    )
    cloud_delta_pct = round(
        float(_usage_idx.loc['casual', 'avg_cloud_cover_pct']) - float(_usage_idx.loc['member', 'avg_cloud_cover_pct']), 2
    )
else:
    temp_delta_f = cloud_delta_pct = 0.0


# ================================================================================
# SECTION 2: WEATHER SENSITIVITY
# ================================================================================

# Coerce rain flag to boolean for reliable rainy vs dry grouping.
weather_an['rainy_day'] = weather_an['rainy_day'].astype(bool)
# Build rainy-vs-dry aggregates with a vectorized pivot.
_weather_pivot = (
    weather_an
    .pivot_table(
        index='rider_type',
        columns='rainy_day',
        values=['ride_count', 'avg_ride_seconds'],
        aggfunc='sum',
        fill_value=0.0,
    )
)

_rainy_rides = _weather_pivot['ride_count'].get(True, pd.Series(index=_weather_pivot.index, dtype=float))
_dry_rides   = _weather_pivot['ride_count'].get(False, pd.Series(index=_weather_pivot.index, dtype=float))
_rainy_dur   = _weather_pivot['avg_ride_seconds'].get(True, pd.Series(index=_weather_pivot.index, dtype=float))
_dry_dur     = _weather_pivot['avg_ride_seconds'].get(False, pd.Series(index=_weather_pivot.index, dtype=float))

_weather_metrics = pd.DataFrame({
    'rainy_rides': _rainy_rides.astype(float),
    'dry_rides':   _dry_rides.astype(float),
    'rainy_duration': _rainy_dur.astype(float),
    'dry_duration':   _dry_dur.astype(float),
}).fillna(0.0)
_weather_metrics['total_rides'] = _weather_metrics['rainy_rides'] + _weather_metrics['dry_rides']

weather_sensitivity_table = (
    _weather_metrics.assign(
        rider_type=_weather_metrics.index,
        rainy_day_ride_share_pct=(
            _weather_metrics['rainy_rides']
            .div(_weather_metrics['total_rides'].replace(0.0, np.nan))
            .fillna(0.0)
            .mul(100.0)
        ),
        rain_vs_dry_ride_count_drop_pct=(
            (_weather_metrics['dry_rides'] - _weather_metrics['rainy_rides'])
            .div(_weather_metrics['dry_rides'].replace(0.0, np.nan))
            .fillna(0.0)
            .mul(100.0)
        ),
        rain_vs_dry_duration_drop_pct=(
            (_weather_metrics['dry_duration'] - _weather_metrics['rainy_duration'])
            .div(_weather_metrics['dry_duration'].replace(0.0, np.nan))
            .fillna(0.0)
            .mul(100.0)
        ),
    )
    .reset_index(drop=True)[[
        'rider_type',
        'rainy_day_ride_share_pct',
        'rain_vs_dry_ride_count_drop_pct',
        'rain_vs_dry_duration_drop_pct',
    ]]
    .round(2)
)
_weather_idx        = weather_sensitivity_table.set_index('rider_type')
casual_rain_share   = float(_weather_idx.loc['casual', 'rainy_day_ride_share_pct'])       if 'casual' in _weather_idx.index else 0.0
member_rain_share   = float(_weather_idx.loc['member', 'rainy_day_ride_share_pct'])       if 'member' in _weather_idx.index else 0.0
casual_ride_drop    = float(_weather_idx.loc['casual', 'rain_vs_dry_ride_count_drop_pct'])if 'casual' in _weather_idx.index else 0.0
member_ride_drop    = float(_weather_idx.loc['member', 'rain_vs_dry_ride_count_drop_pct'])if 'member' in _weather_idx.index else 0.0
casual_duration_drop= float(_weather_idx.loc['casual', 'rain_vs_dry_duration_drop_pct']) if 'casual' in _weather_idx.index else 0.0
member_duration_drop= float(_weather_idx.loc['member', 'rain_vs_dry_duration_drop_pct']) if 'member' in _weather_idx.index else 0.0


# ================================================================================
# SECTION 3: SEASONAL PATTERNS
# ================================================================================

seasonal_peaks_rows       = []
seasonal_campaign_rows    = []
rainy_day_rate_range_rows = []

for rider in rider_types:
    rider_frame = seasonal_an[seasonal_an['rider_type'] == rider]
    if rider_frame.empty:
        continue

    # Sort by volume descending so peak-month tie handling stays deterministic.
    _sorted_frame = rider_frame.sort_values('ride_count', ascending=False)
    peak_row      = _sorted_frame.iloc[0]
    peak_month    = int(peak_row['month'])
    seasonal_peaks_rows.append({
        'rider_type':           rider,
        'peak_month':           peak_month,
        'peak_month_name':      month_name_map.get(peak_month, str(peak_month)),
        'peak_month_ride_count':int(peak_row['ride_count']),
    })

    # Summarize fixed campaign window metrics for each rider type.
    campaign_frame = rider_frame[rider_frame['month'].isin(campaign_months)]
    seasonal_campaign_rows.append({
        'rider_type':                        rider,
        'campaign_window_rides':             int(campaign_frame['ride_count'].sum()),
        'campaign_window_avg_ride_seconds':  round(weighted_mean(campaign_frame, 'avg_ride_seconds'), 2),
        'campaign_window_rainy_day_rate_pct':round(float(campaign_frame['rainy_day_rate'].mean()) * 100.0, 2) if not campaign_frame.empty else 0.0,
    })

    # Capture annual rainy-day-rate range to quantify seasonal variability.
    rainy_day_rate_range_rows.append({
        'rider_type':           rider,
        'min_rainy_day_rate_pct':round(float(rider_frame['rainy_day_rate'].min()) * 100.0, 2),
        'max_rainy_day_rate_pct':round(float(rider_frame['rainy_day_rate'].max()) * 100.0, 2),
    })

seasonal_peaks_table        = pd.DataFrame(seasonal_peaks_rows)
seasonal_campaign_table     = pd.DataFrame(seasonal_campaign_rows)
rainy_day_rate_range_table  = pd.DataFrame(rainy_day_rate_range_rows)

# Build top-3 monthly alternatives with delta-from-peak and tie indicators.
# A tie is flagged when a non-peak month is within 5% of peak volume.
peak_month_alternatives_rows = []
for rider in rider_types:
    rider_frame = seasonal_an[seasonal_an['rider_type'] == rider]
    if rider_frame.empty:
        continue
    _ranked = (
        rider_frame
        .sort_values('ride_count', ascending=False)
        .reset_index(drop=True)
        .head(3)
    )
    _peak_val = int(_ranked['ride_count'].iloc[0])
    for _rank_idx, _r in _ranked.iterrows():
        _delta_pct = round(safe_pct(_peak_val - int(_r['ride_count']), _peak_val), 2)
        peak_month_alternatives_rows.append({
            'rider_type':         rider,
            'rank':               int(_rank_idx) + 1,
            'month':              int(_r['month']),
            'month_name':         month_name_map.get(int(_r['month']), str(int(_r['month']))),
            'ride_count':         int(_r['ride_count']),
            'delta_from_peak_pct':_delta_pct,
            'is_tie':             _delta_pct < 5.0 and int(_rank_idx) > 0,
        })
peak_month_alternatives = pd.DataFrame(peak_month_alternatives_rows)

_seasonal_peaks_idx   = seasonal_peaks_table.set_index('rider_type') if not seasonal_peaks_table.empty else pd.DataFrame()
casual_peak_month_name = (
    str(_seasonal_peaks_idx.loc['casual', 'peak_month_name'])
    if not _seasonal_peaks_idx.empty and 'casual' in _seasonal_peaks_idx.index
    else 'N/A'
)
member_peak_month_name = (
    str(_seasonal_peaks_idx.loc['member', 'peak_month_name'])
    if not _seasonal_peaks_idx.empty and 'member' in _seasonal_peaks_idx.index
    else 'N/A'
)


# ================================================================================
# SECTION 4: TOP 5 START STATIONS BY RIDER TYPE
# ================================================================================

station_share_rows = []
for rider in rider_types:
    rider_total = float(station_stats_an.loc[station_stats_an['rider_type'] == rider, 'ride_count'].sum())
    rider_top5  = float(top5_an.loc[top5_an['rider_type'] == rider, 'ride_count'].sum())
    station_share_rows.append({
        'rider_type':          rider,
        'top5_rides':          int(rider_top5),
        'total_station_rides': int(rider_total),
        'top5_share_pct':      round(safe_pct(rider_top5, rider_total), 2),
    })

station_share_table = pd.DataFrame(station_share_rows)

_station_share_idx = station_share_table.set_index('rider_type') if not station_share_table.empty else pd.DataFrame()
casual_top5_share  = float(_station_share_idx.loc['casual', 'top5_share_pct']) if not station_share_table.empty and 'casual' in _station_share_idx.index else 0.0
member_top5_share  = float(_station_share_idx.loc['member', 'top5_share_pct']) if not station_share_table.empty and 'member' in _station_share_idx.index else 0.0

casual_top_station = 'N/A'
if not top5_an.empty and 'casual' in top5_an['rider_type'].values:
    _casual_top = top5_an[top5_an['rider_type'] == 'casual'].sort_values('ride_count', ascending=False).head(1)
    if not _casual_top.empty:
        casual_top_station = str(_casual_top['start_station'].iloc[0])


# ================================================================================
# SECTION 5: REUSABLE ANALYZE OUTPUTS
# ================================================================================

# Build a compact metric/value table for Share and Act reuse.
_key_metrics_rows = [
    {'section': '[1]', 'metric': 'casual_vs_member_duration_lift_pct',          'value': duration_lift_pct},
    {'section': '[1]', 'metric': 'casual_minus_member_temp_f',                   'value': temp_delta_f},
    {'section': '[1]', 'metric': 'casual_minus_member_cloud_cover_pct_points',   'value': cloud_delta_pct},
]
for _, row in weather_sensitivity_table.iterrows():
    _r = row['rider_type']
    _key_metrics_rows.extend([
        {'section': '[2]', 'metric': f'{_r}_rainy_day_ride_share_pct',        'value': row['rainy_day_ride_share_pct']},
        {'section': '[2]', 'metric': f'{_r}_rain_vs_dry_ride_count_drop_pct', 'value': row['rain_vs_dry_ride_count_drop_pct']},
        {'section': '[2]', 'metric': f'{_r}_rain_vs_dry_duration_drop_pct',   'value': row['rain_vs_dry_duration_drop_pct']},
    ])

analyze_key_metrics_table = pd.DataFrame(_key_metrics_rows)

# Bundle section outputs in a keyed dictionary for downstream visuals.
analyze_section_summaries = {
    '[1]_usage_rider_summary':   usage_rider_summary,
    '[2]_weather_sensitivity':   weather_sensitivity_table,
    '[3]_seasonal_peaks':        seasonal_peaks_table,
    '[3]_campaign_window':       seasonal_campaign_table,
    '[3]_rainy_day_rate_range':  rainy_day_rate_range_table,
    '[3]_peak_month_alternatives': peak_month_alternatives,
    '[4]_station_share':         station_share_table,
}

# Export key metrics in record format for lightweight consumption.
analyze_key_metrics_list = analyze_key_metrics_table.to_dict('records')

# Generate summary text directly from computed metrics (no hardcoded values).
usage_summary_line_1 = (
    f"  [1] Usage    : Casual riders average {duration_lift_pct:.1f}% longer rides than members "
    f"({casual_avg_ride:.0f} s vs {member_avg_ride:.0f} s)."
)
usage_summary_line_2 = (
    f"                 They ride in warmer conditions (+{temp_delta_f:.2f} F) "
    f"and lower cloud cover ({cloud_delta_pct:+.2f} pct pts),"
)
usage_summary_line_3 = "                 showing stronger fair-weather demand."

weather_summary_line_1 = "  [2] Weather  : Casual riders show noticeably higher weather sensitivity than members."
weather_summary_line_2 = (
    f"                 On rainy days casual ride volume drops {casual_ride_drop:.2f}% "
    f"(vs {member_ride_drop:.2f}% for members)"
)
weather_summary_line_3 = (
    f"                 and ride duration drops {casual_duration_drop:.2f}% "
    f"(vs {member_duration_drop:.2f}% for members)."
)
weather_summary_line_4 = "                 Warmer, clearer conditions are the strongest windows for targeting casual riders."

seasonal_summary_line_1 = (
    f"  [3] Seasonal : Casual demand peaks in {casual_peak_month_name} while members peak in {member_peak_month_name}."
)
seasonal_summary_line_2 = "                 The May-Sep window captures the majority of casual volume and longest rides,"
seasonal_summary_line_3 = "                 making it the ideal period for digital campaigns and seasonal passes."

stations_summary_line_1 = (
    f"  [4] Stations : Casual rides concentrate in top 5 stations "
    f"({casual_top5_share:.2f}% of all casual rides)"
)
stations_summary_line_2 = (
    f"                 vs member rides ({member_top5_share:.2f}%), with casuals clustering most at {casual_top_station}."
)


# ================================================================================
# SECTION 6: COMPLEMENTARY DATAFRAMES (AUDIT / REPRODUCIBILITY)
# ================================================================================
# 6a: hourly rider summary
# 6b: daily volume time series
# 6c: day-of-week pattern table
# 6d: monthly full context
# 6e: campaign window comparison

# ================================================================================
# SECTION 6a: AUDIT/REPRODUCIBILITY - HOURLY RIDER SUMMARY
# ================================================================================
if 'hour_of_day' in analysis_df.columns:
    hourly_rider_summary = (
        analysis_df
        .groupby(['hour_of_day', 'member_casual'])
        .agg(**base_agg)
        .reset_index()
        .rename(columns={'member_casual': 'rider_type'})
    )
    hourly_rider_summary['rider_type'] = normalize_rider(hourly_rider_summary['rider_type'])
    hourly_rider_summary = (
        hourly_rider_summary
        .sort_values(['hour_of_day', 'rider_type'])
        .round(2)
        .reset_index(drop=True)
    )
else:
    hourly_rider_summary = pd.DataFrame()

# ================================================================================
# SECTION 6b: AUDIT/REPRODUCIBILITY - DAILY VOLUME TIME SERIES
# ================================================================================
if 'date' in analysis_df.columns:
    daily_volume_ts = (
        analysis_df
        .groupby(['date', 'member_casual'])
        .agg(
            ride_count=      ('ride_id',              'count'),
            avg_ride_seconds=('ride_length_seconds',  'mean'),
            **weather_agg,
            rainy_day_rate=  ('rainy_day',            'mean'),
        )
        .reset_index()
        .rename(columns={'member_casual': 'rider_type'})
    )
    daily_volume_ts['rider_type']      = normalize_rider(daily_volume_ts['rider_type'])
    daily_volume_ts['year_month']      = pd.to_datetime(daily_volume_ts['date']).dt.to_period('M').astype(str)
    daily_volume_ts['day_of_week_num'] = pd.to_datetime(daily_volume_ts['date']).dt.dayofweek
    _dow_n = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
              4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
    daily_volume_ts['day_of_week_name'] = daily_volume_ts['day_of_week_num'].map(_dow_n)
    daily_volume_ts['is_weekend']       = daily_volume_ts['day_of_week_num'].isin([5, 6])
    daily_volume_ts = (
        daily_volume_ts
        .sort_values(['date', 'rider_type'])
        .round(4)
        .reset_index(drop=True)
    )
else:
    daily_volume_ts = pd.DataFrame()

# ================================================================================
# SECTION 6c: AUDIT/REPRODUCIBILITY - DAY-OF-WEEK PATTERN TABLE
# ================================================================================
if 'day_of_week_num' in analysis_df.columns:
    dow_pattern_table = (
        analysis_df
        .groupby(['day_of_week_num', 'day_of_week_name', 'member_casual'])
        .agg(**base_agg)
        .reset_index()
        .rename(columns={'member_casual': 'rider_type'})
    )
    dow_pattern_table['rider_type'] = normalize_rider(dow_pattern_table['rider_type'])
    dow_pattern_table['is_weekend'] = dow_pattern_table['day_of_week_num'].isin([5, 6])
    dow_pattern_table = (
        dow_pattern_table
        .sort_values(['day_of_week_num', 'rider_type'])
        .round(2)
        .reset_index(drop=True)
    )
else:
    dow_pattern_table = pd.DataFrame()

# ================================================================================
# SECTION 6d: AUDIT/REPRODUCIBILITY - MONTHLY FULL CONTEXT
# ================================================================================
if 'year_month' in analysis_df.columns and 'date' in analysis_df.columns:
    _mfc_raw = (
        analysis_df
        .groupby(['year_month', 'member_casual'])
        .agg(
            ride_count=      ('ride_id',              'count'),
            avg_ride_seconds=('ride_length_seconds',  'mean'),
            **weather_agg,
            rainy_day_rate=  ('rainy_day',            'mean'),
            first_date=      ('date',                 'min'),
            last_date=       ('date',                 'max'),
        )
        .reset_index()
        .rename(columns={'member_casual': 'rider_type'})
    )
    _mfc_raw['rider_type']  = normalize_rider(_mfc_raw['rider_type'])
    _mfc_raw['month_name']  = pd.PeriodIndex(_mfc_raw['year_month'], freq='M').strftime('%B')
    _mfc_raw['year']        = pd.PeriodIndex(_mfc_raw['year_month'], freq='M').year
    _mfc_raw['month_num']   = pd.PeriodIndex(_mfc_raw['year_month'], freq='M').month
    # Compute each month's share of annual rides within rider type.
    _rider_year_totals       = _mfc_raw.groupby('rider_type')['ride_count'].transform('sum')
    _mfc_raw['pct_of_rider_year'] = (_mfc_raw['ride_count'] / _rider_year_totals * 100).round(2)
    # Rank monthly volume within each rider type (1 = highest).
    _mfc_raw['volume_rank'] = (
        _mfc_raw.groupby('rider_type')['ride_count']
        .rank(method='min', ascending=False)
        .astype(int)
    )
    monthly_full_context = (
        _mfc_raw[[
            'year_month', 'year', 'month_num', 'month_name', 'rider_type',
            'ride_count', 'pct_of_rider_year', 'volume_rank',
            'avg_ride_seconds', 'avg_temperature_f', 'avg_wind_speed_mph',
            'avg_rain_inches', 'avg_cloud_cover_pct', 'rainy_day_rate',
            'first_date', 'last_date',
        ]]
        .sort_values(['year_month', 'rider_type'])
        .reset_index(drop=True)
    )
    # Round numeric metrics only; keep date and label columns untouched.
    _round_cols = {
        'avg_ride_seconds': 2, 'avg_temperature_f': 2, 'avg_wind_speed_mph': 2,
        'avg_rain_inches': 4, 'avg_cloud_cover_pct': 2, 'rainy_day_rate': 4,
    }
    monthly_full_context = monthly_full_context.round(_round_cols)
else:
    monthly_full_context = pd.DataFrame()

# Publish section 6d output to section summaries for downstream consumers.
analyze_section_summaries['[6d]_monthly_full_context'] = monthly_full_context

# ================================================================================
# SECTION 6e: AUDIT/REPRODUCIBILITY - CAMPAIGN WINDOW COMPARISON
# ================================================================================
if not monthly_full_context.empty and 'casual' in monthly_full_context['rider_type'].values:
    _casual_m = (
        monthly_full_context[monthly_full_context['rider_type'] == 'casual']
        .sort_values(['year_month', 'month_num'])
        [['year_month', 'month_num', 'month_name', 'first_date', 'last_date', 'ride_count', 'pct_of_rider_year']]
        .copy()
    )

    # Derive campaign bounds from observed casual months in this data slice.
    _min_date = pd.to_datetime(_casual_m['first_date']).min()
    _max_date = pd.to_datetime(_casual_m['last_date']).max()
    _actual_ym_sorted = sorted(_casual_m['year_month'].astype(str).unique().tolist())
    _bounded_yms = [
        str(p) for p in pd.period_range(pd.Period(_min_date, 'M'), pd.Period(_max_date, 'M'), freq='M')
    ] if pd.notna(_min_date) and pd.notna(_max_date) else _actual_ym_sorted
    _actual_ym_set = set(_actual_ym_sorted)

    # Materialize fixed-window periods as observed year_month values.
    _fixed_window_yms = [
        ym for ym in _bounded_yms
        if ym in _actual_ym_set and int(ym.split('-')[1]) in campaign_months_fixed
    ]

    # Greedy selection adds highest-volume months until target share is reached.
    _months_by_vol  = _casual_m.sort_values('ride_count', ascending=False)['year_month'].astype(str).tolist()
    _pct_by_ym = dict(zip(_casual_m['year_month'].astype(str), _casual_m['pct_of_rider_year'].astype(float)))
    _derived_months = []
    _cumul_pct      = 0.0
    for _m in _months_by_vol:
        _derived_months.append(_m)
        _row_pct = _pct_by_ym.get(_m, 0.0)
        _cumul_pct += _row_pct
        if _cumul_pct >= CAMPAIGN_TARGET_PCT:
            break
    _derived_months_sorted = sorted(_derived_months)
    _fixed_window_ym_set   = set(_fixed_window_yms)
    _derived_window_ym_set = set(_derived_months_sorted)

    campaign_window_comparison = (
        _casual_m.rename(columns={'ride_count': 'casual_ride_count', 'pct_of_rider_year': 'casual_pct_of_year'})
        .assign(
            in_fixed_window=lambda d: d['year_month'].isin(_fixed_window_ym_set),
            in_derived_window=lambda d: d['year_month'].isin(_derived_window_ym_set),
        )
        .sort_values(['year_month', 'month_num'])
        .reset_index(drop=True)
    )
    _fixed_pct   = campaign_window_comparison.loc[campaign_window_comparison['in_fixed_window'],   'casual_pct_of_year'].sum()
    _derived_pct = campaign_window_comparison.loc[campaign_window_comparison['in_derived_window'], 'casual_pct_of_year'].sum()

    analyze_section_summaries['[6e]_campaign_window_comparison'] = campaign_window_comparison
else:
    campaign_window_comparison = pd.DataFrame()


# ================================================================================
# SECTION 7: VARIABLE LIST SUMMARY
# ================================================================================

# --- Core analysis DataFrames (local analysis copies) ---
# usage_an:               local copy of usage_table used for section 1) rollups.
# weather_an:             local copy of weather_impact_table for section 2) comparisons.
# seasonal_an:            local copy of seasonal_table for section 3) monthly trends.
# station_stats_an:       local copy of station_stats_table for section 4) station totals.
# top5_an:                local copy of top5_stations_table for section 4) station detail.

# --- Section result DataFrames ---
# usage_rider_summary:        rider-level usage summary for section 1).
# weather_sensitivity_table:  rainy-vs-dry comparison table for section 2).
# seasonal_peaks_table:       highest-volume month per rider type for section 3).
# seasonal_campaign_table:    May-Sep campaign summary per rider type for section 3).
# rainy_day_rate_range_table: min/max rainy-day rate per rider type for section 3).
# peak_month_alternatives:    top-3 months per rider type with tie flag for section 3).
# station_share_table:        top-5 station concentration summary for section 4).
# analyze_key_metrics_table:  compact metric/value table for downstream sharing.
# analyze_section_summaries:  dict of all section output DataFrames (includes complementary).

# --- Complementary / reproducibility DataFrames ---
# hourly_rider_summary:       ride volume and duration by hour of day and rider type.
#                             Exposes commute vs leisure timing.
# daily_volume_ts:            daily time series with year_month, DOW, weather context.
#                             Use for continuity checks and anomaly detection.
# dow_pattern_table:          ride counts and duration by full day-of-week (0=Mon ... 6=Sun).
#                             Tests whether weekend binary adequately captures patterns.
# monthly_full_context:       year-aware monthly summary with first/last date, pct_of_rider_year,
#                             volume_rank. Primary reference for reproducibility audits.
#                             Also in analyze_section_summaries['[6d]_monthly_full_context'].
# campaign_window_comparison: fixed May-Sep window vs data-derived window side-by-side.
#                             Shows % casual volume captured by each method.
#                             Also in analyze_section_summaries['[6e]_campaign_window_comparison'].

# --- Helper functions ---
# normalize_rider():  standardizes rider labels for reliable filtering.
# safe_pct():         computes percentages safely when denominators can be zero.
# weighted_mean():    ride-count-weighted averages.
# show_section():     prints section headers and formatted plain-text tables.

# --- Config and grouping variables ---
# required_tables:         upstream tables required before this cell can run.
# missing_tables:          missing required tables, if any.
# month_name_map:          month number -> name, derived from actual year_month data.
# campaign_months_fixed:   hardcoded [5,6,7,8,9] window used in core analysis tables.
# campaign_months:         alias for campaign_months_fixed (keeps Share cell unchanged).
# CAMPAIGN_TARGET_PCT:     threshold (%) for data-driven campaign window derivation; default 70.
# rider_types:             normalized rider labels reused across all sections.

# --- Key scalar metrics ---
# casual_avg_ride / member_avg_ride:  average ride duration by rider type.
# duration_lift_pct:                  % lift in casual duration vs member.
# temp_delta_f:                       casual - member average temperature.
# cloud_delta_pct:                    casual - member average cloud cover.
# casual/member_rain_share:           share of rides on rainy days.
# casual/member_ride_drop:            rain-vs-dry ride-count decline.
# casual/member_duration_drop:        rain-vs-dry duration decline.
# casual/member_peak_month_name:      peak month name per rider type.
# casual/member_top5_share:           top-5 station concentration rates.
# casual_top_station:                 top casual start station by ride count.

# --- Optional export ---
# Uncomment below to persist complementary DataFrames as CSV files.
# -------------------------------------------------------


# ================================================================================
# SECTION 8: OPTIONAL EXPORT (UNCOMMENT TO PERSIST COMPLEMENTARY DFS)
# ================================================================================
# import pathlib
#
# _export_csv_dir = pathlib.Path(
#     '/mnt/RepoRetLabs/code/jupyter-notebooks/'
#     'case-study_bike-share-success/exports/comprehensive_dataframes/csv'
# )
# _export_csv_dir.mkdir(parents=True, exist_ok=True)
#
# _complementary_exports = {
#     'hourly_rider_summary':        hourly_rider_summary,
#     'daily_volume_ts':             daily_volume_ts,
#     'dow_pattern_table':           dow_pattern_table,
#     'monthly_full_context':        monthly_full_context,
#     'campaign_window_comparison':  campaign_window_comparison,
#     'peak_month_alternatives':     peak_month_alternatives,
# }
# for _name, _df in _complementary_exports.items():
#     if not _df.empty:
#         _out = _export_csv_dir / f'{_name}.csv'
#         _df.to_csv(_out, index=False)
#         print(f"  Exported: {_out.name}  ({len(_df):,} rows)")

In [6]:
# ================================================================================
# SECTION 7: FINAL ANALYZE OUTPUT DISPLAY
# Renders technical section outputs, summary lines, and audit/reproducibility tables.
# ================================================================================

# Warn if fixed campaign months are not present in the observed data window.
if _missing_campaign_months:
    print(f"  WARNING: campaign month(s) missing from data: {_missing_campaign_months}")

# ----------------------------------------------------------------
# [1] Usage output block
# Shows rider-level rollup table and key deltas used in summary logic.
# ----------------------------------------------------------------
show_section('[1] Usage Differences by Rider Type', usage_rider_summary, fmt={
    'total_rides':         '{:,.0f}',
    'avg_ride_seconds':    '{:.0f} s',
    'avg_temperature_f':   '{:.1f} F',
    'avg_cloud_cover_pct': '{:.1f}%',
})
print(f"\n  Ride-length lift (casual vs member) : {duration_lift_pct:+.2f}%")
print(f"  Temperature delta (casual - member) : {temp_delta_f:+.2f} F")
print(f"  Cloud cover delta (casual - member) : {cloud_delta_pct:+.2f} percentage points")

# ----------------------------------------------------------------
# [2] Weather sensitivity output block
# Prints comparative rainy-vs-dry impacts and tactical weather notes.
# ----------------------------------------------------------------
show_section('[2] Weather Sensitivity by Rider Type', weather_sensitivity_table, fmt={
    'rainy_day_ride_share_pct':        '{:.2f}%',
    'rain_vs_dry_ride_count_drop_pct': '{:.2f}%',
    'rain_vs_dry_duration_drop_pct':   '{:.2f}%',
})
print("\n  Weather targeting notes:")
print(f"  - Casual rides skew warmer than member rides by {temp_delta_f:+.2f} F.")
print(f"  - Casual rides occur under lower cloud cover by {abs(cloud_delta_pct):.2f} percentage points.")
print(f"  - Rainy days reduce casual ride volume by {casual_ride_drop:.2f}% vs {member_ride_drop:.2f}% for members.")
print(f"  - Rainy-day ride duration drops {casual_duration_drop:.2f}% for casual vs {member_duration_drop:.2f}% for members.")
print(f"  - Rainy-day ride share: {casual_rain_share:.2f}% casual, {member_rain_share:.2f}% member.")

# ----------------------------------------------------------------
# [3] Seasonal output block
# Displays peak timing, alternatives, campaign window, and rainy-day range.
# ----------------------------------------------------------------
show_section('[3] Seasonal Peaks by Rider Type', seasonal_peaks_table, fmt={
    'peak_month_ride_count': '{:,.0f}',
})
show_section('[3] Top-3 Peak Month Alternatives', peak_month_alternatives, fmt={
    'ride_count':         '{:,.0f}',
    'delta_from_peak_pct':'{:.2f}%',
})
show_section('[3] May-Sep Campaign Window (Fixed)', seasonal_campaign_table, fmt={
    'campaign_window_rides':             '{:,.0f}',
    'campaign_window_avg_ride_seconds':  '{:.0f} s',
    'campaign_window_rainy_day_rate_pct':'{:.2f}%',
})
show_section('[3] Annual Rainy-Day Rate Range', rainy_day_rate_range_table, fmt={
    'min_rainy_day_rate_pct': '{:.2f}%',
    'max_rainy_day_rate_pct': '{:.2f}%',
})

# ----------------------------------------------------------------
# [4] Station output block
# Prints top stations by rider type and concentration summary.
# ----------------------------------------------------------------
for rider, rider_frame in top5_an.groupby('rider_type'):
    show_section(
        f'[4] Top 5 Start Stations - {rider.title()}',
        rider_frame[['start_station', 'ride_count', 'avg_temperature_f', 'avg_cloud_cover_pct']].reset_index(drop=True),
        fmt={
            'ride_count':          '{:,.0f}',
            'avg_temperature_f':   '{:.1f} F',
            'avg_cloud_cover_pct': '{:.1f}%',
        }
    )

show_section('[4] Top-5 Station Concentration', station_share_table, fmt={
    'top5_rides':          '{:,.0f}',
    'total_station_rides': '{:,.0f}',
    'top5_share_pct':      '{:.2f}%',
})

# ----------------------------------------------------------------
# Consolidated plain-text summary block
# Uses prebuilt summary lines from the Analyze script core.
# ----------------------------------------------------------------
print_block_header('Summary of Analyze Phase', rule_char='=')
print(usage_summary_line_1)
print(usage_summary_line_2)
print(usage_summary_line_3)
print(weather_summary_line_1)
print(weather_summary_line_2)
print(weather_summary_line_3)
print(weather_summary_line_4)
print(seasonal_summary_line_1)
print(seasonal_summary_line_2)
print(seasonal_summary_line_3)
print(stations_summary_line_1)
print(stations_summary_line_2)
print('=' * OUTPUT_WIDTH)
print("  Casual riders are leisure-focused, weather-sensitive, and location-driven —")
print("  strong targets for membership conversion via hyper-local, fair-weather digital campaigns.")
print('=' * OUTPUT_WIDTH)

# ----------------------------------------------------------------
# [6a-6e] Audit/Reproducibility output blocks
# Displays complementary tables when available; prints skip status otherwise.
# ----------------------------------------------------------------

# [6a] Hourly rider summary preview.
if not hourly_rider_summary.empty:
    show_section(
        '[Audit/Reproducibility 6a] Hourly rider summary (hourly_rider_summary) — first 10 rows',
        hourly_rider_summary[['hour_of_day', 'rider_type', 'ride_count', 'avg_ride_seconds']].head(10).reset_index(drop=True),
        fmt={'ride_count': '{:,.0f}', 'avg_ride_seconds': '{:.2f} s'},
    )
else:
    show_section(
        '[Audit/Reproducibility 6a] Hourly rider summary (hourly_rider_summary)',
        pd.DataFrame([{'status': "skipped: 'hour_of_day' not found in analysis_df."}]),
    )

# [6b] Daily volume time-series coverage check.
if not daily_volume_ts.empty:
    show_section(
        '[Audit/Reproducibility 6b] Daily volume time series (daily_volume_ts)',
        pd.DataFrame([{
            'rows': len(daily_volume_ts),
            'unique_dates': daily_volume_ts['date'].nunique(),
        }]),
        fmt={'rows': '{:,.0f}', 'unique_dates': '{:,.0f}'},
    )
else:
    show_section(
        '[Audit/Reproducibility 6b] Daily volume time series (daily_volume_ts)',
        pd.DataFrame([{'status': "skipped: 'date' not found in analysis_df."}]),
    )

# [6c] Day-of-week behavior profile.
if not dow_pattern_table.empty:
    show_section(
        '[Audit/Reproducibility 6c] Day-of-week pattern (dow_pattern_table)',
        dow_pattern_table[['day_of_week_name', 'rider_type', 'ride_count', 'avg_ride_seconds', 'is_weekend']].reset_index(drop=True),
        fmt={'ride_count': '{:,.0f}', 'avg_ride_seconds': '{:.2f} s'},
    )
else:
    show_section(
        '[Audit/Reproducibility 6c] Day-of-week pattern (dow_pattern_table)',
        pd.DataFrame([{'status': "skipped: 'day_of_week_num' not found in analysis_df."}]),
    )

# [6d] Month-level context for ranking and coverage.
if not monthly_full_context.empty:
    show_section(
        '[Audit/Reproducibility 6d] Monthly full context (monthly_full_context)',
        monthly_full_context[[
            'year_month', 'month_name', 'rider_type', 'ride_count',
            'pct_of_rider_year', 'volume_rank', 'first_date', 'last_date'
        ]].reset_index(drop=True),
        fmt={'ride_count': '{:,.0f}', 'pct_of_rider_year': '{:.2f}%'},
    )
else:
    show_section(
        '[Audit/Reproducibility 6d] Monthly full context (monthly_full_context)',
        pd.DataFrame([{'status': "skipped: 'year_month' or 'date' not found in analysis_df."}]),
    )

# [6e] Fixed-window vs derived-window campaign comparison.
if not campaign_window_comparison.empty:
    show_section(
        '[Audit/Reproducibility 6e] Campaign window comparison (summary)',
        pd.DataFrame([{
            'casual_date_range': f'{_min_date.date()} to {_max_date.date()}',
            'fixed_window_pct': _fixed_pct,
            'derived_window_pct': _derived_pct,
            'fixed_periods': ', '.join(_fixed_window_yms),
            'derived_periods': ', '.join(_derived_months_sorted),
        }]),
        fmt={'fixed_window_pct': '{:.1f}%', 'derived_window_pct': '{:.1f}%'},
    )
    show_section(
        '[Audit/Reproducibility 6e] Campaign Window Comparison (detail)',
        campaign_window_comparison[['year_month', 'month_name', 'casual_pct_of_year', 'in_fixed_window', 'in_derived_window']].reset_index(drop=True),
        fmt={'casual_pct_of_year': '{:.2f}%'},
    )
else:
    show_section(
        '[Audit/Reproducibility 6e] Campaign window comparison (campaign_window_comparison)',
        pd.DataFrame([{'status': 'skipped: monthly_full_context empty or no casual rows.'}]),
    )


# ================================================================================
# SECTION 8: DYNAMIC MARKDOWN SUMMARY
# Converts computed metrics into a stakeholder-ready markdown narrative.
# ================================================================================
from IPython.display import HTML, Markdown, display

# Guard clause: ensure all required summary metrics exist before rendering markdown.
required_metrics = [
    'duration_lift_pct', 'casual_avg_ride', 'member_avg_ride',
    'temp_delta_f', 'cloud_delta_pct',
    'casual_ride_drop', 'member_ride_drop',
    'casual_duration_drop', 'member_duration_drop',
    'casual_rain_share', 'member_rain_share',
    'casual_peak_month_name', 'member_peak_month_name',
    'casual_top5_share', 'member_top5_share', 'casual_top_station',
]

missing_metrics = [m for m in required_metrics if m not in globals()]
if missing_metrics:
    raise NameError(
        "Missing required Analyze metrics. Run Analyze cells first. Missing: "
        + ", ".join(missing_metrics)
    )


# Helper: sign-aware phrase for ride-duration comparisons.
def _duration_phrase(pct):
    if pct > 0.05:
        return f"{pct:.1f}% longer rides"
    if pct < -0.05:
        return f"{abs(pct):.1f}% shorter rides"
    return "nearly identical ride lengths"


# Helper: sign-aware phrase for temperature comparisons.
def _temp_phrase(delta_f):
    if delta_f > 0.05:
        return f"warmer conditions (+{delta_f:.1f} F)"
    if delta_f < -0.05:
        return f"cooler conditions ({delta_f:.1f} F)"
    return "similar temperature conditions"


# Helper: sign-aware phrase for cloud-cover comparisons.
def _cloud_phrase(delta_pct_points):
    if delta_pct_points < -0.05:
        return f"lower cloud cover ({abs(delta_pct_points):.2f} percentage points)"
    if delta_pct_points > 0.05:
        return f"higher cloud cover (+{delta_pct_points:.2f} percentage points)"
    return "similar cloud cover"


# Helper: qualitative weather-sensitivity label based on gaps between rider types.
def _weather_sensitivity_phrase(casual_drop, member_drop, casual_dur_drop, member_dur_drop):
    ride_gap = casual_drop - member_drop
    dur_gap = casual_dur_drop - member_dur_drop
    if ride_gap > 0.1 or dur_gap > 0.1:
        return "higher weather sensitivity"
    if ride_gap < -0.1 or dur_gap < -0.1:
        return "lower weather sensitivity"
    return "similar weather sensitivity"


# Helper: qualitative rainy-day-share intensity label.
def _rain_share_phrase(casual_share, member_share):
    highest_share = max(casual_share, member_share)
    if highest_share < 10:
        return "remains low overall"
    if highest_share < 20:
        return "remains moderate overall"
    return "is substantial overall"


# Build reusable phrase fragments consumed by the markdown template.
row_count = len(analysis_df) if 'analysis_df' in globals() else None
row_text = f"{row_count:,} rows" if row_count is not None else "current rows"

usage_phrase = _duration_phrase(duration_lift_pct)
temp_phrase = _temp_phrase(temp_delta_f)
cloud_phrase = _cloud_phrase(cloud_delta_pct)
weather_phrase = _weather_sensitivity_phrase(
    casual_ride_drop, member_ride_drop, casual_duration_drop, member_duration_drop
)
rain_share_phrase = _rain_share_phrase(casual_rain_share, member_rain_share)

# Build the seasonal sentence with robust fallbacks for missing or tied peak months.
if str(casual_peak_month_name).strip().upper() == 'N/A' or str(member_peak_month_name).strip().upper() == 'N/A':
    seasonal_sentence = (
        "Peak-month timing differs across rider types, and the May through September "
        "window captures the majority of casual volume and longest rides."
    )
elif casual_peak_month_name == member_peak_month_name:
    seasonal_sentence = (
        f"Both rider types peak in **{casual_peak_month_name}**, and the May through "
        "September window captures the majority of casual volume and longest rides."
    )
else:
    seasonal_sentence = (
        f"Casual demand peaks in **{casual_peak_month_name}** while members peak in "
        f"**{member_peak_month_name}**. The May through September window captures "
        "the majority of casual volume and longest rides."
    )

# Use the measured top station when available; otherwise provide a generic fallback label.
top_station_text = (
    casual_top_station
    if str(casual_top_station).strip().upper() != 'N/A' and str(casual_top_station).strip()
    else 'the highest-volume casual start station'
)

# Compose the final markdown report from computed metrics and dynamic phrase fragments.
summary_md = f"""## Analysis Summary

The analysis of the cleaned, weather-enriched dataset ({row_text}) reveals clear, tangible differences between casual and member riders across usage, weather, seasonality, and geography.

**Key Comparative Insights**

- **[1] Usage**  
  Casual riders are more likely to take **{usage_phrase}** than members ({casual_avg_ride:,.0f} s vs {member_avg_ride:,.2f} s), and are more likely to ride in {temp_phrase} and with {cloud_phrase}.

- **[2] Weather Sensitivity**  
  Casual riders show {weather_phrase} than members. On rainy days casual ride volume drops {casual_ride_drop:.2f}% and ride duration drops {casual_duration_drop:.2f}% (compared with {member_ride_drop:.2f}% and {member_duration_drop:.2f}% for members). Rainy-day ride share {rain_share_phrase} ({casual_rain_share:.2f}% casual vs {member_rain_share:.2f}% member).

- **[3] Seasonal Patterns**  
  {seasonal_sentence}

- **[4] Top 5 Start Stations**  
  Casual rides are more concentrated in the top 5 start stations (**{casual_top5_share:.2f}%** of all casual rides) than member rides (**{member_top5_share:.2f}%**), with casuals clustering most at **{top_station_text}**.

**Metric Equations (Reproducibility Reference)**

- **Ride Length Lift Percent**  
  `((casual_avg_ride_seconds - member_avg_ride_seconds) / member_avg_ride_seconds) * 100`

- **Rainy-Day Ride Count Drop Percent**  
  `((dry_day_ride_count - rainy_day_ride_count) / dry_day_ride_count) * 100`

- **Rainy-Day Ride Duration Drop Percent**  
  `((dry_day_avg_ride_seconds - rainy_day_avg_ride_seconds) / dry_day_avg_ride_seconds) * 100`

- **Peak Month by Rider Type**  
  `peak_month = month where ride_count is maximum`

- **Top 5 Station Share Percent**  
  `(top5_station_ride_count / total_station_ride_count) * 100`

- **Temperature Difference (Casual - Member)**  
  `casual_avg_temperature_f - member_avg_temperature_f`

- **Cloud Cover Difference (Casual - Member)**  
  `casual_avg_cloud_cover_percent - member_avg_cloud_cover_percent`
"""

# Render markdown directly in notebook output for stakeholder-facing readability.
display(Markdown(summary_md))


------------------------------------------------------------------------
  [1] Usage Differences by Rider Type
------------------------------------------------------------------------
rider_type total_rides avg_ride_seconds avg_temperature_f avg_cloud_cover_pct
    casual   1,207,570           1034 s            62.9 F               47.8%
    member   2,133,167            689 s            58.1 F               52.2%

  Ride-length lift (casual vs member) : +50.12%
  Temperature delta (casual - member) : +4.80 F
  Cloud cover delta (casual - member) : -4.47 percentage points

------------------------------------------------------------------------
  [2] Weather Sensitivity by Rider Type
------------------------------------------------------------------------
rider_type rainy_day_ride_share_pct rain_vs_dry_ride_count_drop_pct rain_vs_dry_duration_drop_pct
    casual                    8.15%                          91.12%                         3.22%
    member                    8.66%  

## Analysis Summary

The analysis of the cleaned, weather-enriched dataset (3,340,737 rows) reveals clear, tangible differences between casual and member riders across usage, weather, seasonality, and geography.

**Key Comparative Insights**

- **[1] Usage**  
  Casual riders are more likely to take **50.1% longer rides** than members (1,034 s vs 688.79 s), and are more likely to ride in warmer conditions (+4.8 F) and with lower cloud cover (4.47 percentage points).

- **[2] Weather Sensitivity**  
  Casual riders show higher weather sensitivity than members. On rainy days casual ride volume drops 91.12% and ride duration drops 3.22% (compared with 90.52% and 2.27% for members). Rainy-day ride share remains low overall (8.15% casual vs 8.66% member).

- **[3] Seasonal Patterns**  
  Casual demand peaks in **August** while members peak in **September**. The May through September window captures the majority of casual volume and longest rides.

- **[4] Top 5 Start Stations**  
  Casual rides are more concentrated in the top 5 start stations (**8.34%** of all casual rides) than member rides (**4.31%**), with casuals clustering most at **DuSable Lake Shore Dr & Monroe St**.

**Metric Equations (Reproducibility Reference)**

- **Ride Length Lift Percent**  
  `((casual_avg_ride_seconds - member_avg_ride_seconds) / member_avg_ride_seconds) * 100`

- **Rainy-Day Ride Count Drop Percent**  
  `((dry_day_ride_count - rainy_day_ride_count) / dry_day_ride_count) * 100`

- **Rainy-Day Ride Duration Drop Percent**  
  `((dry_day_avg_ride_seconds - rainy_day_avg_ride_seconds) / dry_day_avg_ride_seconds) * 100`

- **Peak Month by Rider Type**  
  `peak_month = month where ride_count is maximum`

- **Top 5 Station Share Percent**  
  `(top5_station_ride_count / total_station_ride_count) * 100`

- **Temperature Difference (Casual - Member)**  
  `casual_avg_temperature_f - member_avg_temperature_f`

- **Cloud Cover Difference (Casual - Member)**  
  `casual_avg_cloud_cover_percent - member_avg_cloud_cover_percent`


## 5. Share

**Objective**  
Communicate the Analyze findings through clear, visual reporting that helps stakeholders formulate marketing strategies to convert casual riders into annual members.

**What the Share code does**  
This section does not recalculate metrics. Instead, it presents validated Analyze outputs through a stakeholder-ready visual report built from interactive charts, source tables, captions, and interpretation blocks.

**Step-by-step Share workflow**  
1. **Validate Share inputs**  
   Confirms required Analyze tables, columns, and scalar metrics exist before any visual is rendered.  
2. **Resolve seasonal timeline source**  
   Uses `monthly_full_context` first, then the published Analyze copy in `analyze_section_summaries['[6d]_monthly_full_context']`, and finally `seasonal_an` as a fallback.  
3. **Standardize display formatting**  
   Applies shared title, caption, source-table, insight-summary, and separator helpers so each visual section follows the same layout.  
4. **Render [1] Usage Differences**  
   Builds a 3-panel comparison of ride duration, temperature, and cloud cover by rider type.  
5. **Render [2] Weather Sensitivity**  
   Reshapes weather metrics into grouped comparisons that show rainy-day exposure and performance drops by rider type.  
6. **Render [3] Seasonal Patterns**  
   Uses a year-month ordered line chart to preserve chronology and highlights the fixed May-September campaign window.  
7. **Render [4] Top 5 Start Stations**  
   Uses detailed top-station output when available, and otherwise falls back to a summary concentration-share view.  
8. **Publish a final Share summary**  
   Produces a consolidated markdown handoff block that summarizes campaign implications from all four visuals.

**Output structure used in each visual section**  
1. Section title  
2. Interactive Plotly chart  
3. Chart-source data table  
4. One-sentence caption  
5. Insight summary block (`What`, `Why`, `Action`, `KPI`)  
6. Visual separator before the next section

**Data Visualization Sources**  
- `analyze_section_summaries`: primary source for section-level Share visuals.  
- `analyze_key_metrics_table`: compact metric table used for headline comparisons and summary values.  
- `monthly_full_context`: year-aware monthly timeline table used for the seasonal trend view.  
- `analyze_section_summaries['[6d]_monthly_full_context']`: published monthly timeline output reused by downstream visuals.  
- `year_month`, `month_name`, `month_num`: timeline fields used to label and order monthly visual outputs.  
- `seasonal_an` and `year_month_list`: fallback seasonal timeline inputs when `monthly_full_context` is unavailable.  
- `monthly_full_context`: fixed May-September campaign window used to highlight the target period.  
- `top5_an`: preferred detailed station-level source for the Top 5 Start Stations visual.
  
*Note: No metric re-calculation occurs in this section; Share only consumes outputs prepared in Analyze*.

In [7]:
# ================================================================================
# SHARE PHASE - VISUALIZATION LAYER
# ================================================================================

# Purpose: Build clean, stakeholder-ready interactive charts from Analyze outputs only.
# No new calculations — only consume published variables.

# Approved data sources (from Analyze):
# `analyze_section_summaries`: primary source for section-level Share visuals.
# `analyze_key_metrics_table`: compact metric table used for headline comparisons and summary values.
# `monthly_full_context`: year-aware monthly timeline table used for the seasonal trend view.
# `analyze_section_summaries['[6d]_monthly_full_context']`: published monthly timeline output reused by downstream visuals.
# `year_month`, `month_name`, `month_num`: timeline fields used to label and order monthly visual outputs.
# `seasonal_an` and `year_month_list`: fallback seasonal timeline inputs when `monthly_full_context` is unavailable.
# `campaign_months`: fixed May-September campaign window used to highlight the target period.

# Visual order (matches markdown): Include 12-month visualized data.
# 1. Usage Differences
# 2. Weather Sensitivity
# 3. Seasonal Patterns
# 4. Top 5 Start Stations

# The chart implementation below uses Plotly for notebook-native interactivity.
# Keep colors consistent: casual = orange, member = blue.
# Use one-sentence captions that state the business takeaway without recalculating metrics.

In [8]:
# ================================================================================
# SECTION 0: IMPORTS AND DEPENDENCY CHECK
# Loads Share display utilities and Plotly with a clear setup error when missing.
# ================================================================================
from IPython.display import HTML, Markdown, display

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError(
        "Plotly is not installed in the active environment. Install it with `%pip install plotly` and rerun this cell."
    ) from exc

# ================================================================================
# SECTION 1: SHARE INPUT VALIDATION
# Ensures Analyze outputs, required columns, and scalar metrics are available.
# ================================================================================
required_share_sections = {
    '[1]_usage_rider_summary': ['rider_type', 'total_rides', 'avg_ride_seconds', 'avg_temperature_f', 'avg_cloud_cover_pct'],
    '[2]_weather_sensitivity': [
        'rider_type',
        'rainy_day_ride_share_pct',
        'rain_vs_dry_ride_count_drop_pct',
        'rain_vs_dry_duration_drop_pct',
    ],
    '[4]_station_share': ['rider_type', 'top5_rides', 'total_station_rides', 'top5_share_pct'],
}

required_scalars = [
    'duration_lift_pct',
    'temp_delta_f',
    'cloud_delta_pct',
    'casual_ride_drop',
    'member_ride_drop',
    'casual_duration_drop',
    'member_duration_drop',
    'casual_rain_share',
    'member_rain_share',
    'casual_peak_month_name',
    'member_peak_month_name',
    'casual_top5_share',
    'member_top5_share',
    'casual_top_station',
]

if 'analyze_section_summaries' not in globals():
    raise NameError("Missing `analyze_section_summaries`. Run the Analyze cells before running Share.")

missing_sections = [key for key in required_share_sections if key not in analyze_section_summaries]
if missing_sections:
    raise KeyError(
        "Missing Share source section(s): " + ", ".join(missing_sections) + ". Run the Analyze cells first."
    )

for section_key, section_columns in required_share_sections.items():
    section_df = analyze_section_summaries[section_key]
    missing_columns = [column for column in section_columns if column not in section_df.columns]
    if missing_columns:
        raise KeyError(
            f"Share source `{section_key}` is missing required columns: {missing_columns}"
        )

missing_scalars = [name for name in required_scalars if name not in globals()]
if missing_scalars:
    raise NameError(
        "Missing Share scalar metrics. Run Analyze output cells first. Missing: " + ", ".join(missing_scalars)
    )

# ================================================================================
# SECTION 2: SOURCE RESOLUTION AND DISPLAY CONFIG
# Resolves seasonal timeline source priority and defines shared display helpers.
# ================================================================================
seasonal_source = pd.DataFrame()
if 'monthly_full_context' in globals() and isinstance(monthly_full_context, pd.DataFrame) and not monthly_full_context.empty:
    seasonal_source = monthly_full_context.copy()
elif '[6d]_monthly_full_context' in analyze_section_summaries and not analyze_section_summaries['[6d]_monthly_full_context'].empty:
    seasonal_source = analyze_section_summaries['[6d]_monthly_full_context'].copy()
elif 'seasonal_an' in globals() and isinstance(seasonal_an, pd.DataFrame) and not seasonal_an.empty:
    seasonal_source = seasonal_an.copy()
    seasonal_source['month_name'] = seasonal_source['month'].map(month_name_map)
    seasonal_source['year_month'] = seasonal_source['month'].apply(
        lambda month_num: next(
            (year_month for year_month in year_month_list if int(year_month.split('-')[1]) == int(month_num)),
            f"month-{int(month_num):02d}"
        )
    )
    seasonal_source['month_num'] = seasonal_source['month']
else:
    raise NameError(
        "No seasonal timeline source is available. Expected `monthly_full_context`, "
        "`analyze_section_summaries['[6d]_monthly_full_context']`, or `seasonal_an`."
    )

if 'campaign_months' not in globals() or not campaign_months:
    raise NameError("Missing `campaign_months`. Run the Analyze cells before running Share.")

color_map = {'casual': "#2ba2f2", 'member': "#272727"}
label_map = {'casual': 'Casual', 'member': 'Member'}


def _normalize_rider_labels(frame, column='rider_type'):
    normalized = frame.copy()
    normalized[column] = normalized[column].astype(str).str.strip().str.lower()
    return normalized


def _display_chart_title(text):
    display(Markdown(f"## {text}"))


def _display_caption(text):
    display(Markdown(f"*{text}*"))


def _display_chart_data_table(title, frame):
    display(Markdown(f"#### {title}"))
    display(frame)


def _display_chart_report(section_title, what_text, why_text, kpi_text):
    report_md = f"""#### {section_title} Insight Summary
- **What it shows:** {what_text}
- **Why it matters:** {why_text}
- **KPI to monitor:** {kpi_text}
"""
    display(Markdown(report_md))


def _display_segment_separator():
    display(HTML("<div style='margin: 12px 0;'><hr style='border: 0; border-top: 1px solid #d0d0d0;'></div>"))


# ================================================================================
# SECTION 3: [1] USAGE DIFFERENCES
# Compares rider groups across duration, temperature, and cloud-cover behavior.
# ================================================================================
usage_vis = _normalize_rider_labels(analyze_section_summaries['[1]_usage_rider_summary'])
usage_metrics = [
    ('avg_ride_seconds', 'Average Ride Duration', 'Seconds', '{:,.0f}'),
    ('avg_temperature_f', 'Average Temperature at Ride Start', 'F', '{:.1f}'),
    ('avg_cloud_cover_pct', 'Average Cloud Cover at Ride Start', 'Percent', '{:.1f}'),
]

fig_usage = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[metric_title for _, metric_title, _, _ in usage_metrics],
    horizontal_spacing=0.08,
    shared_yaxes=False,
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]],
)

usage_index = usage_vis.set_index('rider_type')
for column_index, (metric_key, metric_title, axis_label, _) in enumerate(usage_metrics, start=1):
    for rider_key in ['casual', 'member']:
        metric_value = float(usage_index.loc[rider_key, metric_key]) if rider_key in usage_index.index else 0.0
        fig_usage.add_trace(
            go.Bar(
                x=[label_map[rider_key]],
                y=[metric_value],
                marker_color=color_map[rider_key],
                name=label_map[rider_key],
                legendgroup=rider_key,
                showlegend=(column_index == 1),
                customdata=[[metric_key, axis_label]],
                hovertemplate=(
                    '<b>%{x}</b><br>'
                    'Metric: %{customdata[0]}<br>'
                    'Value: %{y:.2f}<br>'
                    'Unit: %{customdata[1]}<br>'
                    '<extra></extra>'
                ),
            ),
            row=1,
            col=column_index,
        )
    fig_usage.update_yaxes(title_text=axis_label, row=1, col=column_index)
    fig_usage.update_xaxes(title_text='Rider Type', row=1, col=column_index)

fig_usage.update_layout(
    barmode='group',
    height=300,
    legend_title='Rider Type',
    template='plotly_white',
)
_display_chart_title('[1] Usage Differences')
fig_usage.show()

usage_table_out = usage_vis.copy()
usage_table_out['rider_type'] = usage_table_out['rider_type'].map(label_map)
usage_table_out = usage_table_out.rename(columns={
    'rider_type': 'Rider Type',
    'total_rides': 'Total Rides',
    'avg_ride_seconds': 'Avg Ride Seconds',
    'avg_temperature_f': 'Avg Temperature (F)',
    'avg_cloud_cover_pct': 'Avg Cloud Cover (%)',
})
usage_table_out = usage_table_out.sort_values('Rider Type').reset_index(drop=True)
_display_chart_data_table('[1] Chart Data - Usage Differences', usage_table_out)

_display_caption(
    f"Casual riders take {duration_lift_pct:.1f}% longer rides than members and do so in warmer, less cloudy conditions."
)
_display_chart_report(
    '[1] Usage Differences',
    what_text=(
        f"Casual riders post a +{duration_lift_pct:.1f}% ride-duration lift versus members, "
        f"alongside a +{temp_delta_f:.2f} F temperature gap and {cloud_delta_pct:+.2f} cloud-cover-point difference."
    ),
    why_text=(
        "Longer rides under warmer and clearer conditions indicate casual demand is primarily leisure-oriented "
        "rather than routine commuting."
    ),
    kpi_text='Casual-to-member conversion rate on warm, low-cloud days.'
)
_display_segment_separator()

# ================================================================================
# SECTION 4: [2] WEATHER SENSITIVITY
# Reshapes weather-impact metrics and compares rainy-day behavior by rider type.
# ================================================================================
weather_vis = _normalize_rider_labels(analyze_section_summaries['[2]_weather_sensitivity'])
weather_long = weather_vis.melt(
    id_vars=['rider_type'],
    value_vars=[
        'rainy_day_ride_share_pct',
        'rain_vs_dry_ride_count_drop_pct',
        'rain_vs_dry_duration_drop_pct',
    ],
    var_name='metric',
    value_name='value',
)

weather_metric_labels = {
    'rainy_day_ride_share_pct': 'Rainy-Day Ride Share (%)',
    'rain_vs_dry_ride_count_drop_pct': 'Ride Volume Drop vs Dry Days (%)',
    'rain_vs_dry_duration_drop_pct': 'Ride Duration Drop vs Dry Days (%)',
}
weather_long['metric_label'] = weather_long['metric'].map(weather_metric_labels)
weather_long['rider_label'] = weather_long['rider_type'].map(label_map)

fig_weather = px.bar(
    weather_long,
    x='metric_label',
    y='value',
    color='rider_label',
    barmode='group',
    color_discrete_map={'Casual': color_map['casual'], 'Member': color_map['member']},
    labels={'metric_label': '', 'value': 'Percent', 'rider_label': 'Rider Type'},
    template='plotly_white',
)
fig_weather.update_traces(
    customdata=weather_long[['metric', 'rider_type']],
    hovertemplate=(
        '<b>%{x}</b><br>'
        'Rider Type: %{customdata[1]}<br>'
        'Percent: %{y:.2f}%<br>'
        '<extra></extra>'
    ),
)
fig_weather.update_layout(height=300, legend_title='Rider Type')
_display_chart_title('[2] Weather Sensitivity')
fig_weather.show()

weather_table_out = weather_long[['rider_label', 'metric_label', 'value']].copy()
weather_table_out = weather_table_out.rename(columns={
    'rider_label': 'Rider Type',
    'metric_label': 'Metric',
    'value': 'Percent Value',
})
weather_table_out['Percent Value'] = weather_table_out['Percent Value'].round(2)
_display_chart_data_table('[2] Chart Data - Weather Sensitivity', weather_table_out)

_display_caption(
    "Rain sharply reduces demand for both rider groups, but casual riders remain the more weather-sensitive segment in both volume and duration."
)
_display_chart_report(
    '[2] Weather Sensitivity',
    what_text=(
        f"Rainy-day impacts are stronger for casual riders (volume drop {casual_ride_drop:.2f}%, duration drop {casual_duration_drop:.2f}%) "
        f"than members ({member_ride_drop:.2f}% and {member_duration_drop:.2f}%)."
    ),
    why_text=(
        "Weather volatility directly affects casual demand, so campaign timing and media spend must adapt to forecast conditions."
    ),
    kpi_text='Daily conversion cost split by rainy vs dry conditions.'
)
_display_segment_separator()

# ================================================================================
# SECTION 5: [3] SEASONAL PATTERNS
# Enforces year-month chronology and highlights campaign-window seasonality.
# ================================================================================
seasonal_vis = _normalize_rider_labels(seasonal_source)
required_seasonal_columns = ['rider_type', 'ride_count', 'month_num', 'month_name', 'year_month']
missing_seasonal_columns = [column for column in required_seasonal_columns if column not in seasonal_vis.columns]
if missing_seasonal_columns:
    raise KeyError(
        f"Seasonal source is missing required columns for Share visualization: {missing_seasonal_columns}"
    )

seasonal_vis['month_num'] = seasonal_vis['month_num'].astype(int)
seasonal_vis['year_month'] = seasonal_vis['year_month'].astype(str)
seasonal_vis = seasonal_vis.sort_values(['year_month', 'rider_type']).reset_index(drop=True)
seasonal_vis['timeline_label'] = seasonal_vis['month_name'].astype(str) + ' (' + seasonal_vis['year_month'].astype(str) + ')'
year_month_order = seasonal_vis['year_month'].drop_duplicates().tolist()

fig_seasonal = px.line(
    seasonal_vis,
    x='year_month',
    y='ride_count',
    color='rider_type',
    markers=True,
    color_discrete_map=color_map,
    custom_data=['timeline_label', 'avg_ride_seconds', 'avg_temperature_f', 'rainy_day_rate'],
    labels={'year_month': 'Year-Month', 'ride_count': 'Ride Count', 'rider_type': 'Rider Type'},
    template='plotly_white',
)
fig_seasonal.update_traces(
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Ride Count: %{y:,.0f}<br>'
        'Avg Ride Duration: %{customdata[1]:,.2f} sec<br>'
        'Avg Temperature: %{customdata[2]:.2f} F<br>'
        'Rainy-Day Rate: %{customdata[3]:.2%}<br>'
        '<extra></extra>'
    )
)
campaign_periods = [
    year_month for year_month in year_month_order
    if int(str(year_month).split('-')[1]) in campaign_months
]
if campaign_periods:
    fig_seasonal.add_vrect(
        x0=campaign_periods[0],
        x1=campaign_periods[-1],
        fillcolor='#d9d9d9',
        opacity=0.18,
        line_width=0,
        annotation_text='May-Sep campaign window',
        annotation_position='top left',
    )
fig_seasonal.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=year_month_order,
    tickangle=-45,
)
fig_seasonal.update_layout(height=520, legend_title='Rider Type')
_display_chart_title('[3] Seasonal Patterns')
fig_seasonal.show()

seasonal_table_out = seasonal_vis[[
    'year_month', 'month_name', 'rider_type', 'ride_count',
    'avg_ride_seconds', 'avg_temperature_f', 'rainy_day_rate'
]].copy()
seasonal_table_out['rider_type'] = seasonal_table_out['rider_type'].map(label_map)
seasonal_table_out = seasonal_table_out.rename(columns={
    'year_month': 'Year-Month',
    'month_name': 'Month',
    'rider_type': 'Rider Type',
    'ride_count': 'Ride Count',
    'avg_ride_seconds': 'Avg Ride Seconds',
    'avg_temperature_f': 'Avg Temperature (F)',
    'rainy_day_rate': 'Rainy-Day Rate',
})
seasonal_table_out['Rainy-Day Rate'] = (seasonal_table_out['Rainy-Day Rate'] * 100.0).round(2)
_display_chart_data_table('[3] Chart Data - Seasonal Patterns', seasonal_table_out)

_display_caption(
    f"Seasonal demand builds through summer, with casual riders peaking in {casual_peak_month_name} while members peak in {member_peak_month_name}, making May through September the clearest conversion window."
)
_display_chart_report(
    '[3] Seasonal Patterns',
    what_text=(
        f"Seasonal volume rises into summer; casual riders peak in {casual_peak_month_name} while members peak in {member_peak_month_name}."
    ),
    why_text=(
        "This timing identifies when casual intent is strongest, which is the best period to convert leisure usage into annual memberships."
    ),
    kpi_text='Monthly casual-to-member conversion rate during the May-September window.'
)
_display_segment_separator()

# ================================================================================
# SECTION 6: [4] TOP 5 START STATIONS
# Shows station concentration and falls back to summary mode when detail is missing.
# ================================================================================
if 'top5_an' not in globals() or not isinstance(top5_an, pd.DataFrame) or top5_an.empty:
    station_summary_vis = _normalize_rider_labels(analyze_section_summaries['[4]_station_share'])
    fig_station_summary = px.bar(
        station_summary_vis,
        x='rider_type',
        y='top5_share_pct',
        color='rider_type',
        color_discrete_map=color_map,
        labels={'rider_type': 'Rider Type', 'top5_share_pct': 'Top-5 Share (%)'},
        template='plotly_white',
    )
    fig_station_summary.update_traces(
        customdata=station_summary_vis[['top5_rides', 'total_station_rides']],
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Top-5 Share: %{y:.2f}%<br>'
            'Top-5 Rides: %{customdata[0]:,.0f}<br>'
            'Total Station Rides: %{customdata[1]:,.0f}<br>'
            '<extra></extra>'
        ),
    )
    fig_station_summary.update_layout(height=300, showlegend=False)
    _display_chart_title('[4] Top-5 Station Concentration Share')
    fig_station_summary.show()

    station_summary_out = station_summary_vis.copy()
    station_summary_out['rider_type'] = station_summary_out['rider_type'].map(label_map)
    station_summary_out = station_summary_out.rename(columns={
        'rider_type': 'Rider Type',
        'top5_rides': 'Top-5 Rides',
        'total_station_rides': 'Total Station Rides',
        'top5_share_pct': 'Top-5 Share (%)',
    })
    _display_chart_data_table('[4] Chart Data - Top-5 Station Concentration (Fallback)', station_summary_out)

    _display_caption(
        "Top-station detail was unavailable in the current kernel state, so this fallback view shows only the concentration gap between rider groups."
    )
    _display_chart_report(
        '[4] Top Station Concentration (Fallback)',
        what_text=(
            f"Casual top-5 station concentration is {casual_top5_share:.2f}% versus {member_top5_share:.2f}% for members."
        ),
        why_text=(
            "Even without station-level detail, concentration gaps indicate casual demand is easier to target geographically."
        ),
        kpi_text='Conversion rate by top-station cluster vs non-top stations.'
    )
else:
    station_detail_vis = _normalize_rider_labels(top5_an)
    station_detail_vis['rider_label'] = station_detail_vis['rider_type'].map(label_map)
    station_detail_vis = station_detail_vis.sort_values(['rider_type', 'ride_count'], ascending=[True, True])

    fig_stations = px.bar(
        station_detail_vis,
        x='ride_count',
        y='start_station',
        color='rider_label',
        facet_col='rider_label',
        orientation='h',
        color_discrete_map={'Casual': color_map['casual'], 'Member': color_map['member']},
        labels={'ride_count': 'Ride Count', 'start_station': '', 'rider_label': 'Rider Type'},
        template='plotly_white',
    )
    fig_stations.update_traces(
        customdata=station_detail_vis[['avg_temperature_f', 'avg_cloud_cover_pct']],
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Ride Count: %{x:,.0f}<br>'
            'Average Temperature: %{customdata[0]:.2f} F<br>'
            'Average Cloud Cover: %{customdata[1]:.2f}%<br>'
            '<extra></extra>'
        ),
    )
    fig_stations.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
    fig_stations.update_layout(height=350, legend_title='Rider Type', margin=dict(l=220))
    _display_chart_title('[4] Top 5 Start Stations by Rider Type')
    fig_stations.show()

    station_detail_out = station_detail_vis[[
        'rider_label', 'start_station', 'ride_count', 'avg_temperature_f', 'avg_cloud_cover_pct'
    ]].copy()
    station_detail_out = station_detail_out.rename(columns={
        'rider_label': 'Rider Type',
        'start_station': 'Start Station',
        'ride_count': 'Ride Count',
        'avg_temperature_f': 'Avg Temperature (F)',
        'avg_cloud_cover_pct': 'Avg Cloud Cover (%)',
    })
    _display_chart_data_table('[4] Chart Data - Top 5 Start Stations by Rider Type', station_detail_out)

    _display_caption(
        f"Casual demand is more geographically concentrated than member demand, with the strongest cluster centered on {casual_top_station}."
    )
    _display_chart_report(
        '[4] Top 5 Start Stations',
        what_text=(
            f"Casual rides are more concentrated in top stations ({casual_top5_share:.2f}% of rides) than members ({member_top5_share:.2f}%), "
            f"with the largest casual cluster at {casual_top_station}."
        ),
        why_text=(
            "Location concentration reduces targeting waste and improves message relevance for conversion campaigns."
        ),
        kpi_text='Station-level membership conversion lift for top casual stations.'
    )

_display_segment_separator()

# ================================================================================
# SECTION 7: CONSOLIDATED SHARE SUMMARY
# Packages chart outcomes into a presentation-ready narrative handoff block.
# ================================================================================
share_summary_md = f"""## Share Report Summary

This report combines chart visuals, chart-level source tables, and interpretation blocks to which supports stakeholder interest and decision-making.

1. **Usage pattern:** Casual riders show longer rides and fair-weather behavior, supporting leisure-focused conversion messaging.
2. **Weather effect:** Rain sensitivity is stronger for casual riders, supporting weather-triggered media pacing.
3. **Seasonal timing:** Casual demand peaks in **{casual_peak_month_name}**, reinforcing May-September campaign concentration.
4. **Geographic focus:** Casual demand concentration at top stations supports high-efficiency geo-targeted offers.
"""
display(Markdown(share_summary_md))

## [1] Usage Differences

#### [1] Chart Data - Usage Differences

,Rider Type,Total Rides,Avg Ride Seconds,Avg Temperature (F),Avg Cloud Cover (%)
0,Casual,1207570,1034.00,62.86,47.77
1,Member,2133167,688.79,58.06,52.24


*Casual riders take 50.1% longer rides than members and do so in warmer, less cloudy conditions.*

#### [1] Usage Differences Insight Summary
- **What it shows:** Casual riders post a +50.1% ride-duration lift versus members, alongside a +4.80 F temperature gap and -4.47 cloud-cover-point difference.
- **Why it matters:** Longer rides under warmer and clearer conditions indicate casual demand is primarily leisure-oriented rather than routine commuting.
- **KPI to monitor:** Casual-to-member conversion rate on warm, low-cloud days.


...

#### [4] Chart Data - Top 5 Start Stations by Rider Type

,Rider Type,Start Station,Ride Count,Avg Temperature (F),Avg Cloud Cover (%)
4,Casual,DuSable Lake Shore Dr & North Blvd,16132,63.09,39.67
3,Casual,Michigan Ave & Oak St,17495,61.85,42.36
2,Casual,Streeter Dr & Grand Ave,18793,58.17,46.01
1,Casual,Navy Pier,22665,64.48,38.92
0,Casual,DuSable Lake Shore Dr & Monroe St,25641,70.12,48.63
9,Member,Clark St & Elm St,15626,55.18,50.61
8,Member,Clinton St & Madison St,17096,60.55,55.28
7,Member,Canal St & Madison St,17837,58.77,57.05
6,Member,Clinton St & Washington Blvd,19882,59.47,55.88
5,Member,Kingsbury St & Kinzie St,21474,64.40,54.08


*Casual demand is more geographically concentrated than member demand, with the strongest cluster centered on DuSable Lake Shore Dr & Monroe St.*

#### [4] Top 5 Start Stations Insight Summary
- **What it shows:** Casual rides are more concentrated in top stations (8.34% of rides) than members (4.31%), with the largest casual cluster at DuSable Lake Shore Dr & Monroe St.
- **Why it matters:** Location concentration reduces targeting waste and improves message relevance for conversion campaigns.
- **KPI to monitor:** Station-level membership conversion lift for top casual stations.


## Share Report Summary

This report combines chart visuals, chart-level source tables, and interpretation blocks to which supports stakeholder interest and decision-making.

1. **Usage pattern:** Casual riders show longer rides and fair-weather behavior, supporting leisure-focused conversion messaging.
2. **Weather effect:** Rain sensitivity is stronger for casual riders, supporting weather-triggered media pacing.
3. **Seasonal timing:** Casual demand peaks in **August**, reinforcing May-September campaign concentration.
4. **Geographic focus:** Casual demand concentration at top stations supports high-efficiency geo-targeted offers.


## 6. Act

**Objective**  
Translate the Analyze findings into concrete recommendations that the marketing team can use to convert casual riders into annual members.

**Key Recommendations**

- **[1] Usage Differences**  
  Prioritize fair-weather membership offers that highlight value for longer leisure rides, especially on warm and clear days.

- **[2] Weather Sensitivity**  
  Implement weather-triggered digital campaigns: increase budget and conversion messaging on favorable days and reduce spend during heavy rain.

- **[3] Seasonal Patterns**  
  Front-load acquisition efforts in May–July and intensify messaging around peak casual months (August) using station-level and weather-aware targeting.

- **[4] Top 5 Start Stations**  
  Run localized membership promotions at high-concentration casual stations, starting with DuSable Lake Shore Dr & Monroe St.

**Implementation & Measurement Plan**
- Launch the warm-season campaign in May 2026, establishing a baseline casual-to-member conversion rate.
- Monitor the following KPIs monthly: casual-to-member conversion rate, ride frequency of new members, and revenue per converted rider.
- Schedule a 90-day review with the marketing team to evaluate impact and refine targeting strategies.

These recommendations are directly derived from the data patterns identified in Analyze and visualized in Share.

**Conclusion**  
This analysis provides a clear, data-driven foundation for Cyclistic’s casual-to-member conversion strategy. The notebook is fully reproducible and can be easily upgraded to auto-run for automatic monthly reporting with minimal human touch.